# Phase 1: Data Ingestion and Cleaning, point-in-time Nifty 500 universe

**Project:** Fama–French 5-factor model (MKT, SMB, HML, RMW, CMA) on the Indian market.

**What this notebook produces** (inputs for Phases 2–5):

| Output (`data/processed/`) | Contents |
|---|---|
| `universe.csv` | Point-in-time universe: the Nifty-500-equivalent members at each June formation date |
| `monthly_panel.csv` | Stock × month: total return, Rf, excess return, shares, ME (₹ cr), previous month's ME, formation date, `in_universe`, sector, `is_financial` |
| `characteristics_annual.csv` | One row per universe member per June formation: June ME, B/M, OP, asset growth, eligibility helpers, flags |
| `monthly_prices.csv` | Stock × month: unadjusted close, total return, traded value, trading days, eligibility helpers |
| `rf_monthly.csv` | Monthly risk-free rate, known at the start of the month |
| `run_manifest.json` | What this run used (dates, snapshot, counts), for traceability |

**Design rules (classical FF, Indian data)**
- **No survivorship bias:** each June, the universe is the **top 500 NSE-listed companies by full market cap** from NSE's official market-cap ranking files, the Nifty 500's core selection rule (its extra liquidity/F&O eligibility rules are not replicated; see Limitations). Stocks that later delist or merge stay in the data for the years they were members.
- **Prices for every stock** come from NSE daily bhavcopies, so delisted stocks are covered.
- **Corporate actions** (splits, bonuses, rights issues, demergers, mergers) are applied to returns and share counts.
- **Latest data by default:** the data cutoff is the last complete month. Every run writes a manifest.

**The only things that change when the data are refreshed** (no code changes): one new row in `MCAP_RANKINGS` each year when NSE publishes a new ranking file, and rows in the corporate-action tables in Step 5.3 if a check asks for them.

**First run:** Step 3 reads all ~3,900 bhavcopies once (a few minutes; later runs read only new days) and Step 8 downloads Screener pages for the stocks not yet cached (about 2–3 seconds per stock).

In [1]:
# ==========================================================
# Imports
# ==========================================================
import os
import re
import json
import time
import shutil
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

# Optional libraries, only needed when something must be downloaded:
#   nselib   (NSE bhavcopies, corporate actions)   -> %pip install nselib
#   yfinance (share counts)                        -> %pip install yfinance
#   xlrd     (reads the 2016 .xls ranking file)    -> %pip install xlrd
#   bs4      (Screener pages)                      -> %pip install beautifulsoup4
from bs4 import BeautifulSoup

## Step 0: Configuration

Every run setting is in this one cell. Environment variables let a CI/CD pipeline override them without editing code:

| Setting | Default | Override |
|---|---|---|
| `END_DATE` (data cutoff) | last complete month | `FF_END_DATE=2026-09-30` |
| `SCREENER_SNAPSHOT` (fundamentals folder) | newest snapshot younger than 90 days, else today | `FF_SCREENER_SNAPSHOT=2026-10-14` |
| `CLEAN_DERIVED` (wipe derived folders first) | `False` | `FF_CLEAN_DERIVED=1` |
| `UNIVERSE_SIZE` | 500 | `FF_UNIVERSE_SIZE` (testing only) |

**`MCAP_RANKINGS`** lists NSE's official market-cap ranking files ([NSE page](https://www.nseindia.com/static/regulations/listing-compliance/nse-market-capitalisation-all-companies)). **Add one row each year** when NSE publishes a new file. That is a data change, not a code change.

In [2]:
# ==========================================================
# STEP 0: Configuration
# ==========================================================

# ---- Dates ----
RUN_DATE   = date.today()
END_DATE   = pd.Timestamp(os.environ.get(
    "FF_END_DATE", (RUN_DATE.replace(day=1) - timedelta(days=1)).isoformat())).date()   # last complete month
START_DATE = date(2011, 1, 1)          # first price date downloaded

# ---- Model rules ----
UNIVERSE_SIZE         = int(os.environ.get("FF_UNIVERSE_SIZE", 500))   # Nifty-500-equivalent
UNIVERSE_BUFFER       = 150            # extra ranked stocks loaded, in case top-ranked ones did not trade
FIRST_FORMATION_YEAR  = 2016           # first June with a ranking file AND fundamentals (Screener: FY2015+)
FORMATION_MONTH       = 6              # portfolios formed at end of June, held July -> next June
LAG_MONTHS            = 3              # months from fiscal year-end until annual results are public
KEEP_SERIES           = ["EQ", "BE", "BZ"]          # BE/BZ = trade-to-trade (often distressed), must be kept
SERIES_RANK           = {"EQ": 0, "BE": 1, "BZ": 2} # if a stock trades in two series on a day, keep EQ

# ---- Housekeeping ----
CLEAN_DERIVED         = os.environ.get("FF_CLEAN_DERIVED", "0") == "1"
SCREENER_MAX_AGE_DAYS = 90             # reuse a Screener snapshot younger than this

# ---- Paths ----
PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "data"

RAW_DIR      = DATA_DIR / "raw"                       # downloaded data: code never deletes these
UNIVERSE_DIR = RAW_DIR / "universe"
MCAP_DIR     = UNIVERSE_DIR / "nse_mcap"              # NSE market-cap ranking files
BHAV_DIR     = RAW_DIR / "bhavcopy"                   # NSE daily bhavcopies (whole market)
CA_DIR       = RAW_DIR / "corporate_actions"          # NSE corporate actions, one file per year
SHARES_DIR   = RAW_DIR / "shares_yf"                  # yfinance share counts
RF_DIR       = RAW_DIR / "rf"
TRI_DIR      = RAW_DIR / "external" / "nse_tri"       # Nifty total-return indices (checks only)
SCREENER_ROOT = RAW_DIR / "screener_html"

INTERIM_DIR   = DATA_DIR / "interim"                  # derived, rebuilt from raw
PROCESSED_DIR = DATA_DIR / "processed"

if CLEAN_DERIVED:
    for d in [INTERIM_DIR, PROCESSED_DIR]:
        if d.exists():
            shutil.rmtree(d)

for d in [UNIVERSE_DIR, MCAP_DIR, BHAV_DIR, CA_DIR, SHARES_DIR, RF_DIR, TRI_DIR,
          SCREENER_ROOT, INTERIM_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Screener snapshot: newest one younger than SCREENER_MAX_AGE_DAYS, else a new one dated today ----
snapshots = sorted(p.name for p in SCREENER_ROOT.glob("20??-??-??") if p.is_dir() and any(p.glob("*.html")))
if os.environ.get("FF_SCREENER_SNAPSHOT"):
    SCREENER_SNAPSHOT = os.environ["FF_SCREENER_SNAPSHOT"]
elif snapshots and (RUN_DATE - date.fromisoformat(snapshots[-1])).days <= SCREENER_MAX_AGE_DAYS:
    SCREENER_SNAPSHOT = snapshots[-1]
else:
    SCREENER_SNAPSHOT = RUN_DATE.isoformat()

SCREENER_DIR = SCREENER_ROOT / SCREENER_SNAPSHOT
SCREENER_DIR.mkdir(parents=True, exist_ok=True)

# ---- NSE market-cap ranking files: (as-of date, type, URL) ----
# type: "top"     = ranked list, may have no market-cap column (2016-2019)
#       "point"   = market cap of all companies on the as-of date
#       "average" = average market cap over the 6 months ending on the as-of date
NSE_FILES = "https://nsearchives.nseindia.com/web/sites/default/files/inline-files/"
MCAP_RANKINGS = [
    ("2016-03-31", "top",     NSE_FILES + "Top500_BRR_3103016_2.xls"),
    ("2017-03-31", "top",     NSE_FILES + "TOP500_BRR_31032017_2.xlsx"),
    ("2018-03-31", "top",     NSE_FILES + "TOP500_BRR_31032018_2.xlsx"),
    ("2019-03-31", "top",     NSE_FILES + "Top1000_31032019_1.xlsx"),
    ("2019-12-31", "point",   NSE_FILES + "MCAP31122019_0.xlsx"),
    ("2020-03-31", "point",   NSE_FILES + "MCAP_31032020_2.xlsx"),
    ("2020-12-31", "point",   NSE_FILES + "MCAP31122020_1.xlsx"),
    ("2021-03-31", "point",   NSE_FILES + "MCAP31032021_2.xlsx"),
    ("2021-12-31", "point",   NSE_FILES + "MCAP31122021_1.xlsx"),
    ("2022-03-31", "point",   NSE_FILES + "MCAP31032022_1.xlsx"),
    ("2022-12-31", "point",   NSE_FILES + "MCAP31122022_3.xlsx"),
    ("2023-03-31", "point",   NSE_FILES + "MCAP31032023_0.xlsx"),
    ("2023-12-31", "point",   NSE_FILES + "MCAP31122023_0.xlsx"),
    ("2024-03-28", "point",   NSE_FILES + "MCAP28032024.xlsx"),
    ("2024-12-31", "average", NSE_FILES + "Average%20MCAP_July2024ToDecember%202024%20%281%29.xlsx"),
    ("2025-12-31", "average", "https://nsearchives.nseindia.com//web/mediaattachment/2026-01/"
                              "Average_MCAP_July2025ToDecember2025_20260102201101.xlsx"),
]

SYMBOL_CHANGE_URL = "https://nsearchives.nseindia.com/content/equities/symbolchange.csv"
NIFTY500_LIST_URL = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"


In [3]:
# ---- Small helpers used everywhere ----
pd_major, pd_minor = (int(x) for x in pd.__version__.split(".")[:2])
MONTH_END = "ME" if (pd_major, pd_minor) >= (2, 2) else "M"          # pandas month-end code

def parse_dates(s):
    """Dates written in mixed styles ('30-Oct-19', '31 May 2017', '01 June 2016'); pandas 1.x and 2.x."""
    if pd_major >= 2:
        return pd.to_datetime(s, format="mixed", dayfirst=True, errors="coerce")
    return pd.to_datetime(s, dayfirst=True, errors="coerce")

def clean_cols(cols):
    """Fixes BOM/quote garbage in NSE headers and lower-cases them."""
    return (pd.Index(cols).astype(str)
            .str.replace("﻿", "", regex=False).str.replace("ï»¿", "", regex=False)
            .str.replace('"', "", regex=False).str.strip().str.lower())


def to_number(col):
    """'1,234.50' -> 1234.5"""
    return pd.to_numeric(col.astype(str).str.replace(",", ""), errors="coerce")


try:
    from nselib.libutil import nse_urlfetch          # browser-like session that NSE accepts
    NSELIB_OK = True
except ImportError:
    NSELIB_OK = False
    print("nselib not installed: nothing will be downloaded from NSE (cached files are used)")


def nse_download(url, path):
    """Download one NSE file. Returns True if saved."""
    if not NSELIB_OK:
        return False
    try:
        r = nse_urlfetch(url)
        if r.status_code == 200 and len(r.content) > 1000:
            path.write_bytes(r.content)
            return True
    except Exception as e:
        print("  download failed:", path.name, "|", type(e).__name__, e)
    return False


print("Run date:", RUN_DATE, "| data cutoff (END_DATE):", END_DATE, "| universe size:", UNIVERSE_SIZE)
print("Screener snapshot:", SCREENER_SNAPSHOT, "| pages cached:", len(list(SCREENER_DIR.glob("*.html"))))
print("CLEAN_DERIVED:", CLEAN_DERIVED, "| pandas", pd.__version__)

Run date: 2026-11-12 | data cutoff (END_DATE): 2026-10-31 | universe size: 500
Screener snapshot: 2026-10-14 | pages cached: 1722
CLEAN_DERIVED: False | pandas 2.1.4


## Step 1: Symbol changes

NSE stocks change ticker over time (e.g. MOTHERSUMI → MOTHERSON, SRTRANSFIN → SHRIRAMFIN, TATAMOTORS → TMPV). Every price, ranking and corporate-action record uses the symbol **of its own date**. `resolve(symbol, date)` follows NSE's symbol-change list (plus `MANUAL_RENAMES` for renames missing from it) forward from that date and returns the stock's **latest symbol**, used as its permanent ID (`symbol` in all outputs). It is date-aware, so a symbol later reused by a different company is not mixed up.

In [4]:
# ==========================================================
# STEP 1: Symbol changes -> permanent stock ID (latest symbol)
# ==========================================================

SYMBOL_CHANGE_FILE = UNIVERSE_DIR / "symbolchange.csv"
if not SYMBOL_CHANGE_FILE.exists() or date.fromtimestamp(SYMBOL_CHANGE_FILE.stat().st_mtime) < END_DATE:
    nse_download(SYMBOL_CHANGE_URL, SYMBOL_CHANGE_FILE)

sc = pd.read_csv(SYMBOL_CHANGE_FILE, encoding="latin-1")
sc.columns = clean_cols(sc.columns)
sc = sc.rename(columns={"old symbol": "old", "new symbol": "new", "effective date": "eff_date"})
sc["old"] = sc["old"].astype(str).str.strip().str.upper()
sc["new"] = sc["new"].astype(str).str.strip().str.upper()
sc["eff_date"] = parse_dates(sc["eff_date"])
assert sc["eff_date"].notna().mean() > 0.95, "Symbol-change dates were not parsed - check the date format"
sc = sc.dropna(subset=["eff_date"]).sort_values("eff_date")

CHANGES = {}                                        # old symbol -> [(effective date, new symbol), ...]
for old, new, eff in zip(sc["old"], sc["new"], sc["eff_date"]):
    CHANGES.setdefault(old, []).append((eff, new))


# Renames missing from NSE's list, found by Step 11 check 5b (data, not code).
# Effective date = first trading day under the new symbol.
MANUAL_RENAMES = [
    # old symbol,   new symbol,   effective date
    ("PHILIPCARB", "PCBL",       "2022-01-13"),   # Phillips Carbon Black -> PCBL
    ("BURGERKING", "RBA",        "2022-02-17"),   # Burger King India -> Restaurant Brands Asia
    ("WABCOINDIA", "ZFCVINDIA",  "2022-04-01"),   # WABCO India -> ZF Commercial Vehicle Control Systems
    ("RUCHI",      "PATANJALI",  "2022-07-13"),   # Ruchi Soya -> Patanjali Foods
]

for old, new, eff in MANUAL_RENAMES:
    CHANGES.setdefault(old, []).append((pd.Timestamp(eff), new))
for old in CHANGES:
    CHANGES[old].sort()
    
def resolve(symbol, when):
    """Latest symbol of the stock that traded as `symbol` on date `when`."""
    sym, t = str(symbol).strip().upper(), pd.Timestamp(when)
    for _ in range(10):                              # at most 10 renames
        nxt = [(e, n) for e, n in CHANGES.get(sym, []) if e > t]
        if not nxt:
            return sym
        t, sym = nxt[0]
    return sym


def resolve_frame(df, sym_col, date_col):
    """Vectorised resolve(): only symbols that ever changed are looked up row by row."""
    out = df[sym_col].astype(str).str.strip().str.upper()
    hit = out.isin(CHANGES.keys())
    if hit.any():
        pairs = df.loc[hit, [sym_col, date_col]].drop_duplicates()
        lookup = {(s, t): resolve(s, t) for s, t in zip(pairs[sym_col], pairs[date_col])}
        out.loc[hit] = [lookup[(s, t)] for s, t in zip(df.loc[hit, sym_col], df.loc[hit, date_col])]
    return out


print("Symbol changes:", len(sc), "| example:", resolve("MOTHERSUMI", "2020-01-01"),
      "|", resolve("SRTRANSFIN", "2020-01-01"))

Symbol changes: 1079 | example: MOTHERSON | SHRIRAMFIN


## Step 2: Universe candidates from NSE's market-cap rankings

**Rule:** for June of year *t*, use the latest ranking file dated on or before 31 March of *t* (public before June, so no look-ahead). Its top `UNIVERSE_SIZE + UNIVERSE_BUFFER` stocks are loaded; the final universe (Step 4) is the top 500 of those that actually traded in the 3 months up to the file date.

Files with a market-cap column also give **exact market-cap anchors** for share counts (Step 9). NSE quotes market cap in ₹ lakh; it is converted to ₹ crore.

In [5]:
# ==========================================================
# STEP 2: NSE market-cap ranking files -> universe candidates
# ==========================================================

def read_ranking(path):
    """One NSE ranking file -> DataFrame(rank, symbol, mcap_cr). Finds the header row itself."""
    raw = pd.read_excel(path, header=None, sheet_name=0)
    header_row = next(i for i in range(min(len(raw), 20))
                      if raw.iloc[i].astype(str).str.strip().str.lower().eq("symbol").any())
    df = raw.iloc[header_row + 1:].copy()
    df.columns = clean_cols(raw.iloc[header_row])

    rank_col = df.columns[0]                                         # "sr. no." or "rank"
    mcap_cols = [c for c in df.columns if "capitali" in c or "mcap" in c]
    out = pd.DataFrame({"rank": pd.to_numeric(df[rank_col], errors="coerce"),
                        "symbol": df["symbol"].astype(str).str.strip().str.upper()})
    
    if mcap_cols:
        raw_mcap = pd.to_numeric(df[mcap_cols[0]], errors="coerce")
        # Units: headers say lakh, but some files are in rupees. Decide from the data: the largest
        # Indian company has always been between Rs 50 thousand crore and Rs 1 crore crore.
        top = raw_mcap.max()
        scale = next((s for s in [1.0, 0.01, 1e-4, 1e-7]          # crore, lakh, thousand, rupees
                      if 5e4 <= top * s <= 1e7), 0.01 if "lakh" in mcap_cols[0] else 1.0)
        out["mcap_cr"] = raw_mcap * scale
        out.attrs["unit_scale"] = scale
    
    else:
        out["mcap_cr"] = np.nan
    
    return out.dropna(subset=["rank"]).query("symbol != 'NAN'").reset_index(drop=True)


rankings = []
for as_of, kind, url in MCAP_RANKINGS:
    path = MCAP_DIR / url.split("/")[-1].replace("%20", " ").replace("%28", "(").replace("%29", ")")
    if not path.exists():
        print("Downloading", path.name)
        if not nse_download(url, path):
            print(f"  -> MISSING: download it in a browser and save it as {path}")
            continue
    
    r = read_ranking(path)
    r["as_of"], r["kind"], r["file"] = pd.Timestamp(as_of), kind, path.name
    print(f"{as_of}  {path.name:60s} unit scale to Rs crore: {r.attrs.get('unit_scale', 'no market cap')}")
    
    r["sid"] = resolve_frame(r.assign(d=pd.Timestamp(as_of)), "symbol", "d")
    rankings.append(r)

rankings = pd.concat(rankings, ignore_index=True)

# Which file decides each June formation?
formation_years = [y for y in range(FIRST_FORMATION_YEAR, END_DATE.year + 1)
                   if pd.Timestamp(y, FORMATION_MONTH, 1) + pd.offsets.MonthEnd(0) <= pd.Timestamp(END_DATE)]
file_for_year = {}
for y in formation_years:
    usable = rankings.loc[rankings["as_of"] <= pd.Timestamp(y, 3, 31), "as_of"]
    if len(usable):
        file_for_year[y] = usable.max()

candidates = pd.concat([
    rankings[(rankings["as_of"] == file_for_year[y]) & (rankings["rank"] <= UNIVERSE_SIZE + UNIVERSE_BUFFER)]
    .assign(formation_date=pd.Timestamp(y, FORMATION_MONTH, 1) + pd.offsets.MonthEnd(0))
    for y in file_for_year
], ignore_index=True)

print(rankings.groupby(["as_of", "kind"]).agg(stocks=("sid", "count"),
                                              with_mcap=("mcap_cr", "count")).to_string())
print("\nFile used for each June formation:")
print({y: str(d.date()) for y, d in file_for_year.items()})
print("Candidate stocks (all years):", candidates["sid"].nunique())

2016-03-31  Top500_BRR_3103016_2.xls                                     unit scale to Rs crore: no market cap
2017-03-31  TOP500_BRR_31032017_2.xlsx                                   unit scale to Rs crore: no market cap
2018-03-31  TOP500_BRR_31032018_2.xlsx                                   unit scale to Rs crore: no market cap
2019-03-31  Top1000_31032019_1.xlsx                                      unit scale to Rs crore: no market cap
2019-12-31  MCAP31122019_0.xlsx                                          unit scale to Rs crore: 1e-07
2020-03-31  MCAP_31032020_2.xlsx                                         unit scale to Rs crore: 0.01
2020-12-31  MCAP31122020_1.xlsx                                          unit scale to Rs crore: 0.01
2021-03-31  MCAP31032021_2.xlsx                                          unit scale to Rs crore: 0.01
2021-12-31  MCAP31122021_1.xlsx                                          unit scale to Rs crore: 0.01
2022-03-31  MCAP31032022_1.xlsx              

## Step 3: Daily prices for every candidate (NSE bhavcopies)

Bhavcopies hold one day's prices for **every** NSE stock, including later-delisted ones. Two formats are read (NSE changed format on 8-Jul-2024). Only series EQ/BE/BZ are kept; if a stock trades in two series on a day, EQ wins. The combined table is cached in `interim/` and only new days are read on later runs.

In [6]:
# ==========================================================
# STEP 3: NSE bhavcopies -> daily prices (cached, incremental)
# ==========================================================
UDIFF_START = date(2024, 7, 8)      # NSE changed the bhavcopy format on this date


def bhavcopy_url(d):
    if d < UDIFF_START:
        mon = d.strftime("%b").upper()
        return (f"https://nsearchives.nseindia.com/content/historical/EQUITIES/"
                f"{d.year}/{mon}/cm{d:%d}{mon}{d.year}bhav.csv.zip")
    return f"https://nsearchives.nseindia.com/content/cm/BhavCopy_NSE_CM_0_0_0_{d:%Y%m%d}_F_0000.csv.zip"


# 3.1 Download missing days (holidays simply have no file)
have = {p.name[:10] for p in BHAV_DIR.glob("*.csv.zip")}
missing_days = [d.date() for d in pd.bdate_range(START_DATE, END_DATE) if d.strftime("%Y-%m-%d") not in have]
for d in missing_days:                               # market holidays never have a file (about 15 a year)
    if nse_download(bhavcopy_url(d), BHAV_DIR / f"{d:%Y-%m-%d}.csv.zip"):
        time.sleep(0.3)
print("Bhavcopy files:", len(list(BHAV_DIR.glob("*.csv.zip"))), "| days tried:", len(missing_days))


Bhavcopy files: 3905 | days tried: 225


In [7]:
# 3.2 Read files -> one table (date, symbol, series, close, turnover)
def read_bhavcopy(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    day = pd.Timestamp(path.name[:10])
    if "TckrSymb" in df.columns:                                   # new (UDiFF) format
        df = df.rename(columns={"TckrSymb": "symbol", "SctySrs": "series",
                                "ClsPric": "close", "TtlTrfVal": "turnover"})
    else:                                                           # old format
        df = df.rename(columns={"SYMBOL": "symbol", "SERIES": "series",
                                "CLOSE": "close", "TOTTRDVAL": "turnover"})
    df = df[["symbol", "series", "close", "turnover"]].copy()
    df["series"] = df["series"].astype(str).str.strip().str.upper()
    df = df[df["series"].isin(KEEP_SERIES)]
    df["symbol"] = df["symbol"].astype(str).str.strip().str.upper()
    df["date"] = day
    return df


BHAV_CACHE = INTERIM_DIR / "bhav_daily.pkl"
bhav = pd.read_pickle(BHAV_CACHE) if BHAV_CACHE.exists() else pd.DataFrame()
last_cached = bhav["date"].max() if len(bhav) else pd.Timestamp("1900-01-01")
new_files = [p for p in sorted(BHAV_DIR.glob("*.csv.zip"))
             if last_cached < pd.Timestamp(p.name[:10]) <= pd.Timestamp(END_DATE)]
if new_files:
    bhav = pd.concat([bhav] + [read_bhavcopy(p) for p in new_files], ignore_index=True)
    for c in ["symbol", "series"]:
        bhav[c] = bhav[c].astype("category")                       # keeps the cache small
    bhav.to_pickle(BHAV_CACHE)
bhav = bhav[bhav["date"] <= pd.Timestamp(END_DATE)]
print("Bhavcopy rows:", len(bhav), "| new files read:", len(new_files),
      "| dates:", bhav["date"].min().date(), "->", bhav["date"].max().date())


Bhavcopy rows: 6952617 | new files read: 0 | dates: 2011-01-03 -> 2026-10-30


In [8]:
# 3.3 Keep candidates only (including their old symbols), permanent ID, one row per stock per day
def final_symbol(sym):
    """Last symbol in a rename chain, ignoring dates (only used to pre-filter rows)."""
    for _ in range(10):
        if sym not in CHANGES:
            return sym
        sym = CHANGES[sym][-1][1]
    return sym

cand_sids = set(candidates["sid"])
wanted = cand_sids | {old for old in CHANGES if final_symbol(old) in cand_sids}
daily = bhav[bhav["symbol"].astype(str).isin(wanted)].copy()
daily["symbol"] = daily["symbol"].astype(str)
daily["series"] = daily["series"].astype(str)
daily["sid"] = resolve_frame(daily, "symbol", "date")
daily = daily[daily["sid"].isin(cand_sids)]
daily["rank"] = daily["series"].map(SERIES_RANK)
daily = (daily.sort_values(["sid", "date", "rank"])
              .drop_duplicates(["sid", "date"], keep="first").drop(columns="rank"))
daily["close"] = pd.to_numeric(daily["close"], errors="coerce")
daily["turnover"] = pd.to_numeric(daily["turnover"], errors="coerce")
daily = daily.dropna(subset=["close"]).reset_index(drop=True)

daily["yesterday_close"] = daily.groupby("sid")["close"].shift(1)
daily["gap_days"] = daily.groupby("sid")["date"].diff().dt.days      # long gap = no trading
daily["price_jump"] = daily["yesterday_close"] / daily["close"]      # > 1 = price fell
LAST_PRICE_DATE = daily["date"].max()
print("Daily rows (candidates):", len(daily), "| stocks:", daily["sid"].nunique(),
      "| last date:", LAST_PRICE_DATE.date())

Daily rows (candidates): 3051039 | stocks: 1070 | last date: 2026-10-30


## Step 4: Final universe

For each June, the universe is the **top `UNIVERSE_SIZE` ranked stocks that traded at least once in the 3 months up to the ranking file's date** (a listed stock that did not trade cannot be held). Members keep their June membership for the whole holding year (July–June), even if they delist during it.

In [9]:
# ==========================================================
# STEP 4: Final universe (top 500 that traded)
# ==========================================================
traded_rows = []
for y, as_of in file_for_year.items():
    window = daily[(daily["date"] > as_of - pd.DateOffset(months=3)) & (daily["date"] <= as_of)]
    traded = set(window["sid"])
    c = candidates[candidates["formation_date"].dt.year == y].sort_values("rank")
    c = c[c["sid"].isin(traded)].drop_duplicates("sid").head(UNIVERSE_SIZE)
    traded_rows.append(c)

universe = pd.concat(traded_rows, ignore_index=True)[
    ["formation_date", "sid", "symbol", "rank", "mcap_cr", "as_of", "file"]]
universe = universe.rename(columns={"symbol": "symbol_then", "mcap_cr": "ranking_mcap_cr",
                                    "as_of": "ranking_as_of", "file": "ranking_file"})
UNIVERSE_SIDS = set(universe["sid"])
universe.to_csv(PROCESSED_DIR / "universe.csv", index=False)

turnover_check = universe.groupby("formation_date")["sid"].apply(set)
yrs = list(turnover_check.index)
print(universe.groupby("formation_date").agg(members=("sid", "count"), max_rank=("rank", "max")).to_string())
print("\nStocks ever in the universe:", len(UNIVERSE_SIDS))
print("Members leaving each year:", {str(yrs[i].year): len(turnover_check.iloc[i - 1] - turnover_check.iloc[i])
                                     for i in range(1, len(yrs))})

daily = daily[daily["sid"].isin(UNIVERSE_SIDS)].reset_index(drop=True)

                members  max_rank
formation_date                   
2016-06-30          500     500.0
2017-06-30          500     500.0
2018-06-30          500     500.0
2019-06-30          500     500.0
2020-06-30          500     500.0
2021-06-30          500     500.0
2022-06-30          500     500.0
2023-06-30          500     500.0
2024-06-30          500     500.0
2025-06-30          500     500.0
2026-06-30          500     500.0

Stocks ever in the universe: 879
Members leaving each year: {'2017': 45, '2018': 65, '2019': 45, '2020': 51, '2021': 49, '2022': 61, '2023': 45, '2024': 63, '2025': 45, '2026': 50}


## Step 5: Corporate actions

**Why:** NSE's `prevclose` is **not** adjusted on split/bonus ex-dates in this data (checked in the first Phase 1 version: 303 of 306 events unadjusted), so every split or bonus would look like a crash. All adjustments are therefore built here from NSE's corporate-actions file, then checked against actual price drops.

| Event | Effect | Source |
|---|---|---|
| Bonus X:Y | share count × (X+Y)/Y | NSE file |
| Face-value split from Rs A to Rs B | share count × A/B (consolidation < 1) | NSE file |
| Dividend | added to the ex-date return (only the dividend part of a combined subject is read; "Purpose Revised" duplicates counted once) | NSE file |
| Rights issue | return factor = cum price ÷ theoretical ex-rights price (CRSP convention; partly-paid treated as fully paid) | NSE file |
| Demerger | ex-day return set to 0 (the drop is value moved to the new company) | NSE file (from ~2019) + manual list |
| Merger into another listed stock | final-month return includes the share-swap value | manual list |
| Missing / wrong events | added or fixed, factor implied by the price drop | manual lists |

Every event is mapped to the **first trading day on or after its ex-date** (within 30 days, so ex-dates inside a trading gap still map).

**Checks, handled automatically and logged in 5.9:** a split/bonus whose price drop is found within 15 days of the NSE ex-date is moved to that day, and one with no matching drop is dropped; a one-day move > 37.5% down or > 66% up that matches a standard split/bonus factor (within 3%) is added as a missing event, and any other is kept as a real move; rights factors > 1.5 are applied only when the price drop confirms them. **Must be 0:** manual demergers not mapped to a trading day. Known cases can still be fixed by hand in the manual tables in 5.3 (data, not code).

In [10]:
# ==========================================================
# STEP 5.1: NSE corporate-actions file (one file per year)
# ==========================================================
for yr in range(START_DATE.year, END_DATE.year + 1):
    path = CA_DIR / f"corporate_actions_{yr}.csv"
    # Past years never change; the cutoff year is re-downloaded until a copy made after the cutoff exists
    if path.exists() and (yr < END_DATE.year or date.fromtimestamp(path.stat().st_mtime) > END_DATE):
        continue
    if not NSELIB_OK:
        continue
    try:
        from nselib import capital_market
        capital_market.corporate_actions_for_equity(
            from_date=f"01-01-{yr}", to_date=f"31-12-{yr}").to_csv(path, index=False)
        time.sleep(1)
    except Exception as e:
        print("Corporate actions", yr, "not downloaded:", type(e).__name__, e)

ca = pd.concat([pd.read_csv(p) for p in sorted(CA_DIR.glob("corporate_actions_*.csv"))], ignore_index=True)
ca.columns = clean_cols(ca.columns)
ca["ex_date"] = pd.to_datetime(ca["exdate"], dayfirst=True, errors="coerce")
ca = ca.dropna(subset=["ex_date"])
ca["symbol"] = ca["symbol"].astype(str).str.strip().str.upper()
ca["sid"] = resolve_frame(ca, "symbol", "ex_date")
print("Corporate-action records:", len(ca), "| years:", ca["ex_date"].dt.year.min(), "->", ca["ex_date"].dt.year.max())

Corporate-action records: 33798 | years: 2011 -> 2026


In [11]:
# ==========================================================
# STEP 5.2: Parsers (with self-tests)
# ==========================================================

def dividend_per_share(subject, face_value):
    """
    'Interim Dividend - Rs 5 Per Share' -> 5.0 ; 'Dividend - 50%' with FV 10 -> 5.0
    Only the dividend part of a combined subject is read
    ('.../Dividend - Rs 10 Per Share/Face Value Split From Rs 10 To Re 1' -> 10, not 21).
    """
    parts = [p for p in str(subject).lower().split("/") if re.search(r"\bdiv(idend)?\b", p)]
    if not parts:                                     # no dividend (also skips 'Sub-Division')
        return 0.0
    text = " / ".join(parts)
    amounts = re.findall(r"(?:\brs|\bre|\binr|₹)\.?\s*-?\s*(\d+(?:\.\d+)?)", text)
    total = sum(float(a) for a in amounts)
    if total == 0:                                    # old style: dividend as % of face value
        pcts = re.findall(r"(\d+(?:\.\d+)?)\s*%", text)
        fv = pd.to_numeric(face_value, errors="coerce")
        if pcts and pd.notna(fv):
            total = sum(float(p) for p in pcts) / 100 * fv
    return total


def parse_share_events(subject):
    """
    Subject -> list of (event_type, share_mult); share_mult = new share count / old share count.
      'Bonus 1:2' -> [('bonus', 1.5)] ;  'From Rs 10/- ... To Re 1/-' -> [('split', 10.0)] ;
      'From Re 1 To Rs 10' (consolidation) -> [('split', 0.1)]
    """
    s = str(subject).lower()
    events = []
    for x, y in re.findall(r"bonus\s*(?:issue)?\s*(\d+)\s*:\s*(\d+)", s):
        if "debenture" in s or "preference" in s:     # not equity shares
            continue
        x, y = float(x), float(y)
        if y > 0:
            events.append(("bonus", (x + y) / y))
    if "split" in s or "sub-div" in s or "consolidat" in s:
        m = re.search(r"from\s*r[se]\.?\s*(\d+(?:\.\d+)?).*?to\s*r[se]\.?\s*(\d+(?:\.\d+)?)", s)
        if m:
            old_fv, new_fv = float(m.group(1)), float(m.group(2))
            if new_fv > 0 and old_fv != new_fv:
                events.append(("split", old_fv / new_fv))
    return events


def parse_rights(subject, face_value):
    """
    'Rights 8:15 @ Premium Rs 40/-' with face value 10 -> (8, 15, 50.0)
    Returns (new_shares, held_shares, issue_price) or None. Issue price = face value + premium.
    """
    s = str(subject).lower()
    if "right" not in s or any(w in s for w in ["debenture", "ncd", "warrant", "preference"]):
        return None
    ratio = re.search(r"rights?\s*(?:issue)?[^0-9:]{0,40}?(\d+)\s*:\s*(\d+)", s)
    if ratio is None:
        return None
    new_sh, held_sh = float(ratio.group(1)), float(ratio.group(2))
    fv = pd.to_numeric(face_value, errors="coerce")
    if re.search(r"@\s*par\b", s) and pd.notna(fv):
        price = fv
    else:
        p = re.search(r"@\s*(premium\s*(?:of\s*)?)?r[se]\.?\s*(\d+(?:\.\d+)?)", s)
        if p is None:
            return None
        price = float(p.group(2))
        if p.group(1):
            if pd.isna(fv):
                return None
            price += fv
    if held_sh <= 0 or new_sh <= 0:
        return None
    return new_sh, held_sh, price


# Self-tests: every line must start with OK
tests = [
    (dividend_per_share("Interim Dividend - Rs 5 Per Share", 10), 5.0),
    (dividend_per_share("Dividend - 50%", 10), 5.0),
    (dividend_per_share("Annual General Meeting/Dividend - Rs 10 Per Share/Face Value Split (Sub-Division) - "
                        "From Rs 10/- Per Share To Re 1/- Per Share", 10), 10.0),
    (dividend_per_share("Dividend - Rs 2 Per Share/Rights 1:5 @ Premium Rs 40/-", 10), 2.0),
    (parse_share_events("Bonus 1:2"), [("bonus", 1.5)]),
    (parse_share_events(" Bonus 1:5/Face Value Split (Sub-Division) - From Rs 10/- Per Share To Rs 2/- Per Share"),
     [("bonus", 1.2), ("split", 5.0)]),
    (parse_share_events("Annual General Meeting/Dividend Rs.3/- Per Share/Scheme Of Arrangement - "
                        "Bonus Debentures 1:1 (Purpose Revised)"), []),
    (parse_share_events("Face Value Consolidation From Re 1/- Per Share To Rs 10/- Per Share"), [("split", 0.1)]),
    (parse_rights("Rights 8:15 @ Premium Rs 40/-", 10), (8.0, 15.0, 50.0)),
    (parse_rights("Rights 2:7 @ Rs 205/-", 10), (2.0, 7.0, 205.0)),
    (parse_rights("Rights 1:6 @ Par", 10), (1.0, 6.0, 10.0)),
    (parse_rights("Rights Issue Of Non Convertible Debentures", 10), None),
]
for got, expected in tests:
    if isinstance(got, list):
        got = [(t, round(f, 4)) for t, f in got]
    print("OK  " if got == expected else "FAIL", got)

OK   5.0
OK   5.0
OK   10.0
OK   2.0
OK   [('bonus', 1.5)]
OK   [('bonus', 1.2), ('split', 5.0)]
OK   []
OK   [('split', 0.1)]
OK   (8.0, 15.0, 50.0)
OK   (2.0, 7.0, 205.0)
OK   (1.0, 6.0, 10)
OK   None


In [12]:
# ==========================================================
# STEP 5.3: Event tables + manual corrections
# ==========================================================

# --- Dividends (Rs per share, per ex-date; "Purpose Revised" repeats counted once) ---
ca["div_ps"] = [dividend_per_share(s, fv) for s, fv in zip(ca["subject"], ca["faceval"])]
dividends = (ca[ca["div_ps"] > 0].drop_duplicates(["sid", "ex_date", "div_ps"])
             .groupby(["sid", "ex_date"], as_index=False)["div_ps"].sum())

# --- Splits / bonuses: total share multiplier, and the split-only part (face-value change) ---
rows = []
for sid, ex, subject in zip(ca["sid"], ca["ex_date"], ca["subject"]):
    for event_type, mult in parse_share_events(subject):
        rows.append({"sid": sid, "ex_date": ex, "event_type": event_type, "share_mult": mult})
ev = pd.DataFrame(rows).drop_duplicates(["sid", "ex_date", "event_type", "share_mult"])
ev["fv_mult"] = np.where(ev["event_type"] == "split", ev["share_mult"], 1.0)
split_events = (ev.groupby(["sid", "ex_date"], as_index=False)
                  .agg(event_type=("event_type", lambda s: "+".join(sorted(s))),
                       share_mult=("share_mult", "prod"), fv_mult=("fv_mult", "prod")))
split_events["source"] = "nse_file"

# --- Rights issues ---
rows = []
for sid, ex, subject, fv in zip(ca["sid"], ca["ex_date"], ca["subject"], ca["faceval"]):
    parsed = parse_rights(subject, fv)
    if parsed is not None:
        rows.append({"sid": sid, "ex_date": ex, "rights_new": parsed[0], "rights_held": parsed[1],
                     "issue_price": parsed[2], "subject": subject})
rights_events = pd.DataFrame(rows, columns=["sid", "ex_date", "rights_new", "rights_held",
                                            "issue_price", "subject"]).drop_duplicates(["sid", "ex_date"])


In [13]:
# ----------------------------------------------------------------------------------------
# MANUAL TABLES (data, not code). Symbol = NSE symbol used ON THAT DATE (latest symbol also works).
# Found by the checks in 5.5-5.8 and a review of large one-day falls.
# ----------------------------------------------------------------------------------------
MANUAL_EVENTS = [            # missing from the NSE file; factor implied by the one-day price drop
    # symbol,       ex_date,      share_mult, type
    ("MANAPPURAM", "2011-06-09",  2.0, "split"),
    ("M&MFIN",     "2013-02-15",  5.0, "split"),
    ("ASIANPAINT", "2013-07-30", 10.0, "split"),
    ("SBIN",       "2014-11-20", 10.0, "split"),
    ("GRANULES",   "2015-03-23", 10.0, "split"),
    ("MOTHERSUMI", "2015-07-23",  1.5, "bonus"),
    ("SOLARINDS",  "2016-07-13",  5.0, "split"),
    ("CAPLIPOINT", "2016-10-19",  5.0, "split"),
    ("KARURVYSYA", "2016-11-17",  5.0, "split"),
    ("CGCL",       "2016-11-23",  5.0, "split"),
    ("KPRMILL",    "2016-11-29",  2.0, "split"),
    ("JSWSTEEL",   "2017-01-04", 10.0, "split"),
    ("DAAWAT",     "2017-02-07", 10.0, "split"),
    ("AJANTPHARM", "2022-06-22",  1.5, "bonus"),
]

FIXED_FACTORS = [            # NSE file has the event but the wrong factor
    # symbol,      ex_date,      correct share_mult
    ("HINDZINC",  "2011-03-07", 10.0),      # file: split x5; same-day bonus 1:1 missing
    ("MINDACORP", "2015-01-05",  2.0),      # file: x10; price drop says x2
]

DEMERGER_DAYS = [            # demergers not labelled in the NSE file (it labels them reliably only from ~2019)
    ("ADANIENT",   "2015-06-03"), ("JSL",        "2015-12-04"), ("CROMPGREAV", "2016-03-15"),
    ("KPIT",       "2019-01-24"), ("IIFL",       "2019-06-14"), ("CENTURYTEX", "2019-10-11"),
    ("TATACHEM",   "2020-03-04"), ("FAIRCHEM",   "2020-08-21"), ("ANANTRAJ",   "2020-10-06"),
    ("GMRINFRA",   "2022-01-11"), ("MOTHERSUMI", "2022-01-14"), ("AARTIIND",   "2022-10-19"),
    ("NMDC",       "2022-10-27"), ("RELIANCE",   "2023-07-20"), ("ITC",        "2025-01-06"),
    ("SIEMENS",    "2025-04-07"), ("STLTECH",    "2025-04-24"), ("ABFRL",      "2025-05-22"),
    ("TATAMOTORS", "2025-10-14"), ("HINDUNILVR", "2025-12-05"), ("VEDL",       "2026-04-30"),
    ("HEGAM",      "2026-09-07"),
]

REAL_MOVES = [               # big one-day falls checked and confirmed as real (kept as they are)
    ("YESBANK", "2020-03-06"),        # RBI moratorium
]

MERGERS = [                  # stock merged INTO another listed stock: final-month return includes the swap
    # target (symbol then), acquirer (symbol then), acquirer shares received per target share
    ("SBT",        "SBIN",       2.2),     # Apr-2017, 22 SBI per 10
    ("SBBJ",       "SBIN",       2.8),     # Apr-2017, 28 SBI per 10
    ("MYSOREBANK", "SBIN",       2.2),     # Apr-2017, 22 SBI per 10
    ("ABIRLANUVO", "GRASIM",     1.5),     # Jul-2017, 3 Grasim (FV 10) per 10 = 15 Grasim (FV 2, after the Oct-2016 split) per 10
    ("CAPF",       "IDFCBANK",  13.9),     # Dec-2018, 139 IDFC Bank per 10
    ("VIJAYABANK", "BANKBARODA", 0.402),   # Apr-2019, 402 per 1000
    ("DENABANK",   "BANKBARODA", 0.110),   # Apr-2019, 110 per 1000
    ("BHARATFIN",  "INDUSINDBK", 0.639),   # Jul-2019, 639 per 1000
    ("GRUH",       "BANDHANBNK", 0.568),   # Oct-2019, 568 per 1000
    ("SYNDIBANK",  "CANBK",      0.158),   # Apr-2020, 158 per 1000
    ("ANDHRABANK", "UNIONBANK",  0.324),   # Apr-2020, 324 per 1000
    ("CORPBANK",   "UNIONBANK",  0.330),   # Apr-2020, 330 per 1000
    ("ALBK",       "INDIANB",    0.115),   # Apr-2020, 115 per 1000
    ("ORIENTBANK", "PNB",        1.150),   # Apr-2020, 1150 per 1000
    ("UNITEDBNK",  "PNB",        0.121),   # Apr-2020, 121 per 1000
    ("GSKCONS",    "HINDUNILVR", 4.39),    # Apr-2020, 4.39 HUL per 1
    ("TATASTLBSL", "TATASTEEL",  1 / 15),  # Nov-2021, 1 Tata Steel per 15
    ("MINDTREE",   "LTI",        0.73),    # Nov-2022, 73 LTI per 100
    ("SHRIRAMCIT", "SRTRANSFIN", 1.55),    # Dec-2022, 1.55 per 1
    ("INOXLEISUR", "PVR",        0.3),     # Feb-2023, 3 PVR per 10
    ("EQUITAS",    "EQUITASBNK", 2.31),    # Feb-2023, 231 per 100
    ("JSLHISAR",   "JSL",        1.95),    # Mar-2023, 195 JSL per 100
    ("HDFC",       "HDFCBANK",   42 / 25), # Jul-2023, 42 HDFC Bank per 25
    ("TATASTLLP",  "TATASTEEL",  67 / 10), # Nov-2023, 67 Tata Steel per 10
    ("UJJIVAN",    "UJJIVANSFB", 11.665),  # May-2024, 116.65 per 10
    ("TCNSBRANDS", "ABFRL",      2.2),     # Sep-2024, 11 ABFRL per 5
    ("TATACOFFEE", "TATACONSUM", 3 / 10),  # Jan-2024, 3 Tata Consumer per 10 Tata Coffee
    ("IDFC",       "IDFCFIRSTB", 1.55),    # Oct-2024, 155 per 100
    ("ISEC",       "ICICIBANK",  0.67),    # Mar-2025, 67 per 100
]

# Not listed (ratio unconfirmed): TV18BRDCST -> NETWORK18 (Oct-2024); its returns stop at the last trade.
# Not listed (ratio unconfirmed): TV18BRDCST -> NETWORK18, TATASTLLP -> TATASTEEL, TATACOFFEE -> TATACONSUM;
# their returns stop at the last trade. Any row whose swap return exceeds +/-25% is skipped automatically.


In [14]:
def manual_frame(rows, cols):
    df = pd.DataFrame(rows, columns=cols)
    df["ex_date"] = pd.to_datetime(df["ex_date"])
    df["sid"] = resolve_frame(df, "symbol", "ex_date")
    return df


# 1. Fix wrong factors (each must change exactly one row)
for sym, dt, mult in FIXED_FACTORS:
    sid = resolve(sym, dt)
    row = (split_events["sid"] == sid) & (split_events["ex_date"] == pd.Timestamp(dt))
    print(f"Fix {sym} {dt}: rows matched = {row.sum()}")          # must be 1
    split_events.loc[row, ["share_mult", "source"]] = [mult, "nse_file, factor fixed"]

# 2. Add missing events (skipped if the NSE file already has that stock and date)
manual = manual_frame(MANUAL_EVENTS, ["symbol", "ex_date", "share_mult", "event_type"])
manual["fv_mult"] = np.where(manual["event_type"] == "split", manual["share_mult"], 1.0)
manual["source"] = "manual (price-implied)"
split_events = (pd.concat([split_events, manual.drop(columns="symbol")], ignore_index=True)
                  .drop_duplicates(["sid", "ex_date"], keep="first")
                  .sort_values(["sid", "ex_date"]).reset_index(drop=True))

# 3. Demergers: manual list + every demerger in the NSE file
dm_manual = manual_frame(DEMERGER_DAYS, ["symbol", "ex_date"]).assign(source="manual")
dm_file = (ca[ca["subject"].str.contains("demerg", case=False, na=False)]
           [["sid", "ex_date"]].drop_duplicates().assign(source="nse_file"))
demergers = pd.concat([dm_manual[["sid", "ex_date", "source"]], dm_file], ignore_index=True)
real_moves = manual_frame(REAL_MOVES, ["symbol", "ex_date"])[["sid", "ex_date"]]

print("\nDividends:", len(dividends), "| split/bonus events:", len(split_events),
      "| rights:", len(rights_events), "| demergers:", len(demergers), "| mergers:", len(MERGERS))
print(split_events["source"].value_counts().to_string())

Fix HINDZINC 2011-03-07: rows matched = 1
Fix MINDACORP 2015-01-05: rows matched = 1

Dividends: 20004 | split/bonus events: 1011 | rights: 314 | demergers: 129 | mergers: 29
source
nse_file                  995
manual (price-implied)     14
nse_file, factor fixed      2


In [15]:
# ==========================================================
# STEP 5.4: Map every event to its first trading day (our stocks only)
# ==========================================================

def to_trading_day(table, max_days=30):
    """Attach trade_date = first trading day on/after ex_date (same stock), plus that day's prices."""
    left = table[table["sid"].isin(UNIVERSE_SIDS)].sort_values("ex_date").reset_index(drop=True)
    right = daily[["sid", "date", "close", "yesterday_close", "gap_days", "price_jump"]].sort_values("date")
    m = pd.merge_asof(left, right, left_on="ex_date", right_on="date", by="sid",
                      direction="forward", tolerance=pd.Timedelta(days=max_days))
    return m.rename(columns={"date": "trade_date"})


split_m = to_trading_day(split_events)
rights_m = to_trading_day(rights_events)
dividends_m = to_trading_day(dividends, max_days=7)
demerger_all = to_trading_day(demergers)
demerger_m = (demerger_all.dropna(subset=["trade_date"]).sort_values("source")       # manual first
                          .drop_duplicates(["sid", "trade_date"], keep="first"))
manual_unmapped = demerger_all[(demerger_all["source"] == "manual") & demerger_all["trade_date"].isna()
                               & (demerger_all["ex_date"] <= LAST_PRICE_DATE)]
real_m = to_trading_day(real_moves)

print("Mapped to a trading day -> splits/bonuses:", split_m["trade_date"].notna().sum(),
      "| rights:", rights_m["trade_date"].notna().sum(), "| dividends:", dividends_m["trade_date"].notna().sum(),
      "| demergers:", len(demerger_m))

Mapped to a trading day -> splits/bonuses: 472 | rights: 95 | dividends: 11488 | demergers: 69


In [16]:
# ==========================================================
# STEP 5.5 CHECK 1: every split/bonus must match the price drop on its trading day
# NSE prevclose is not adjusted, so yesterday's close / today's close is close to share_mult.
# Automatic handling (logged):
#   - drop found within 15 days of the NSE ex-date -> event MOVED to that day (wrong ex-date in the file)
#   - no matching drop at all                      -> event DROPPED (applying it would create a fake jump)
# After a trading gap (> 5 days) the real move in between makes the test meaningless: event kept.
# ==========================================================

def split_status(t):
    jump_vs_mult = t["price_jump"] / t["share_mult"]
    return np.where(t["gap_days"] > 5, "after trading gap",
           np.where(jump_vs_mult.between(0.8, 1.25), "confirmed", "FAILED"))

split_m["status"] = np.where(split_m["trade_date"].isna(), "no trading day", split_status(split_m))
split_log = []

for i in split_m.index[split_m["status"] == "FAILED"]:
    e = split_m.loc[i]
    win = daily[(daily["sid"] == e["sid"]) & daily["date"].between(e["ex_date"] - pd.Timedelta(days=15),
                                                                   e["ex_date"] + pd.Timedelta(days=15))]
    hit = win[(win["price_jump"] / e["share_mult"]).between(0.8, 1.25) & (win["gap_days"] <= 5)]
    if len(hit):
        h = hit.loc[(hit["date"] - e["ex_date"]).abs().idxmin()]
        split_m.loc[i, ["trade_date", "close", "yesterday_close", "gap_days", "price_jump"]] = \
            [h["date"], h["close"], h["yesterday_close"], h["gap_days"], h["price_jump"]]
        split_m.loc[i, "status"] = "moved to matching drop"
    else:
        split_m.loc[i, "status"] = "DROPPED (no matching drop)"
    split_log.append(split_m.loc[i, ["sid", "ex_date", "trade_date", "share_mult", "status", "source"]])

split_m = split_m[split_m["status"] != "DROPPED (no matching drop)"].copy()
print(pd.concat([split_m["status"], pd.Series([r["status"] for r in split_log if "DROPPED" in r["status"]],
                                               dtype=object)]).value_counts().to_string())
if split_log:
    display(pd.DataFrame(split_log))

confirmed                     469
DROPPED (no matching drop)      3
no trading day                  2


,sid,ex_date,trade_date,share_mult,status,source
52,ASHOKA,2013-07-03,2013-07-03,1.5,DROPPED (no matching drop),nse_file
87,JBMA,2014-10-08,2014-10-08,2.0,DROPPED (no matching drop),nse_file
95,MINDACORP,2015-01-05,2015-01-05,2.0,DROPPED (no matching drop),"nse_file, factor fixed"


In [17]:
# ==========================================================
# STEP 5.6 CHECK 2: every big one-day move (fall > 37.5% or rise > 66%) must be explained
# Automatic handling of unexplained moves (logged):
#   - price ratio within 3% of a standard split/bonus factor -> missing split/bonus ADDED (price-implied)
#   - anything else -> kept as a real price move (listed for information)
# ==========================================================
STANDARD_FACTORS = [1.25, 4 / 3, 1.5, 2, 2.5, 3, 4, 5, 10, 20, 0.5, 0.2, 0.1]

def explain_big_moves():
    big = daily[(daily["price_jump"] > 1.6) | (daily["price_jump"] < 0.6)].copy()
    keys = {name: set(zip(t["sid"], t["trade_date"]))
            for name, t in [("split/bonus", split_m), ("demerger", demerger_m),
                            ("rights issue", rights_m), ("real move (listed)", real_m)]}
    
    big_key = list(zip(big["sid"], big["date"]))
    big["explained_by"] = "UNKNOWN"
    big.loc[big["gap_days"] > 30, "explained_by"] = "data gap (monthly return set to NaN)"
    
    for name, k in keys.items():
        big.loc[[x in k for x in big_key], "explained_by"] = name
    return big

big = explain_big_moves()
unk = big[(big["explained_by"] == "UNKNOWN") & (big["gap_days"] <= 5)]
added = []

for _, u in unk.iterrows():
    f = min(STANDARD_FACTORS, key=lambda x: abs(u["price_jump"] / x - 1))
    if abs(u["price_jump"] / f - 1) <= 0.03:
        added.append({"sid": u["sid"], "ex_date": u["date"], "event_type": "auto", "share_mult": f,
                      "fv_mult": f, "source": "auto (price-implied)", "trade_date": u["date"],
                      "close": u["close"], "yesterday_close": u["yesterday_close"],
                      "gap_days": u["gap_days"], "price_jump": u["price_jump"], "status": "auto-added"})
        
if added:
    split_m = pd.concat([split_m, pd.DataFrame(added)], ignore_index=True)
big = explain_big_moves()
big.loc[big["explained_by"] == "UNKNOWN", "explained_by"] = "real move (not reviewed)"

print(big["explained_by"].value_counts().to_string())
print("\nMissing splits/bonuses added from the price drop:")
display(pd.DataFrame(added)[["sid", "trade_date", "share_mult", "price_jump"]] if added else "none")
print("Large one-day moves kept as real (not reviewed):")
display(big.loc[big["explained_by"] == "real move (not reviewed)",
                ["sid", "symbol", "date", "yesterday_close", "close", "price_jump", "gap_days"]])

explained_by
split/bonus                             385
demerger                                 28
real move (not reviewed)                 15
data gap (monthly return set to NaN)      3
rights issue                              1
real move (listed)                        1

Missing splits/bonuses added from the price drop:


,sid,trade_date,share_mult,price_jump
0,ASHOKA,2013-07-03,3.0,2.980952
1,JSL,2015-11-19,2.5,2.555556
2,MASTEK,2015-06-12,3.0,2.940920
3,SINTEX,2017-05-25,4.0,4.027079
4,TRIVENI,2011-05-03,2.5,2.497494


Large one-day moves kept as real (not reviewed):


,sid,symbol,date,yesterday_close,close,price_jump,gap_days
27335,ABIRLANUVO,ABIRLANUVO,2016-01-20,2065.55,882.15,2.341495,1.0
404488,CCAVENUE,INFIBEAM,2018-09-28,200.35,58.45,3.427716,1.0
406470,CCAVENUE,CCAVENUE,2026-10-19,13.83,130.05,0.106344,7.0
585830,DHFL,DHFL,2018-09-21,610.55,350.55,1.741692,2.0
715279,FEL,FRL,2016-05-11,151.05,19.00,7.950000,1.0
1062980,ICIL,ICIL,2016-11-11,799.80,150.05,5.330223,1.0
1103631,IIFL,IIFL,2019-05-30,417.45,199.50,2.092481,1.0
1221999,JBMA,JBMA,2014-10-08,1037.70,248.90,4.169144,1.0
1227060,JETAIRWAYS,JETAIRWAYS,2019-06-18,68.40,40.50,1.688889,1.0
1227062,JETAIRWAYS,JETAIRWAYS,2019-06-20,33.10,62.85,0.526651,1.0


In [18]:
# ==========================================================
# STEP 5.7 CHECK 3: demergers by source; every manual one must map to a trading day
# ==========================================================
print(demerger_m["source"].value_counts().to_string())
display(demerger_m.sort_values("trade_date")[["sid", "ex_date", "trade_date", "price_jump", "gap_days", "source"]])
display(manual_unmapped)          # must be empty

source
nse_file    47
manual      22


,sid,ex_date,trade_date,price_jump,gap_days,source
0,JINDALPOLY,2013-07-17,2013-07-17,1.032412,1.0,nse_file
1,POLARIS,2014-10-09,2014-10-09,1.180952,1.0,nse_file
2,GREENPLY,2014-11-26,2014-11-26,1.169602,1.0,nse_file
3,SFCL,2015-04-23,2015-04-23,1.014212,1.0,nse_file
4,ADANIENT,2015-06-03,2015-06-03,5.804100,1.0,manual
...,...,...,...,...,...,...
80,GENUSPOWER,2026-02-06,2026-02-06,0.967601,1.0,nse_file
82,VEDL,2026-04-30,2026-04-30,2.848831,1.0,manual
83,GUJENERGY,2026-07-02,2026-07-02,1.133144,1.0,nse_file
84,TRIVENI,2026-07-22,2026-07-22,1.711434,1.0,nse_file


,sid,ex_date,source,trade_date,close,yesterday_close,gap_days,price_jump


In [19]:
# ==========================================================
# STEP 5.8: Rights issues -> return factor (used only if <= 1.5 or confirmed by the price drop)
# TERP = (held x cum price + new x issue price) / (held + new); factor = cum price / TERP
# ==========================================================

r = rights_m[rights_m["trade_date"].notna()].copy()
cum = r["yesterday_close"]
terp = (r["rights_held"] * cum + r["rights_new"] * r["issue_price"]) / (r["rights_held"] + r["rights_new"])

r["ret_factor"] = np.where(r["issue_price"] < cum, cum / terp, 1.0)
r["price_confirms"] = (r["price_jump"] / r["ret_factor"]).between(0.8, 1.25) & (r["gap_days"] <= 5)
r["use"] = (r["ret_factor"] <= 1.5) | r["price_confirms"]
rights_ok = r[r["use"]]

print("Rights issues of our stocks:", len(r), "| factor > 1.5:", int((r["ret_factor"] > 1.5).sum()),
      "| of which confirmed by price:", int(((r["ret_factor"] > 1.5) & r["price_confirms"]).sum()))

display(r.sort_values("ret_factor", ascending=False)
        [["sid", "ex_date", "rights_new", "rights_held", "issue_price", "yesterday_close",
          "ret_factor", "price_jump", "use"]].head(10))

Rights issues of our stocks: 95 | factor > 1.5: 5 | of which confirmed by price: 4


,sid,ex_date,rights_new,rights_held,issue_price,yesterday_close,ret_factor,price_jump,use
70,HERITGFOOD,2023-01-20,1.0,1.0,5.0,351.50,1.971950,1.881188,True
19,WHEELS,2014-02-13,51.0,20.0,400.0,1054.55,1.804561,1.406911,False
43,IDEA,2019-03-29,87.0,38.0,12.5,29.00,1.655629,1.589041,True
54,M&MFIN,2020-07-22,1.0,1.0,50.0,227.90,1.640158,1.481313,True
24,NCC,2014-09-18,7.0,6.0,20.0,71.55,1.633849,1.525586,True
0,KARURVYSYA,2011-02-17,2.0,5.0,142.0,530.60,1.264624,1.258987,True
31,KTKBANK,2016-10-24,1.0,2.0,70.0,157.80,1.227697,1.229930,True
72,PNBHOUSING,2023-04-05,29.0,54.0,275.0,540.90,1.207379,1.167872,True
7,GAYAPROJ,2012-02-22,1.0,1.0,112.0,170.25,1.206377,1.205310,True
91,HCC,2025-12-05,277.0,630.0,12.5,25.94,1.187980,1.303518,True


In [20]:
# ==========================================================
# STEP 5.9: Summary and save
# ==========================================================

summary = {
    "split/bonus events confirmed by the price drop":         int((split_m["status"] == "confirmed").sum()),
    "split/bonus events moved to the matching drop":          int((split_m["status"] == "moved to matching drop").sum()),
    "split/bonus events dropped (no price drop)":             sum("DROPPED" in r["status"] for r in split_log),
    "missing splits/bonuses added (price-implied)":           len(added),
    "large one-day moves kept as real (not reviewed)":        int((big["explained_by"] == "real move (not reviewed)").sum()),
    "rights factors > 1.5 not applied (no price confirm.)":   int((~r["use"]).sum()),
    "manual demergers not mapped (MUST be 0)":                int(len(manual_unmapped)),
}

for k, v in summary.items():
    flag = ("PASS" if v == 0 else "CHECK") if "MUST" in k else ""
    print(f"{k:58s} {v:5d}  {flag}")
CHECKS_STEP5 = summary

split_m.to_csv(INTERIM_DIR / "split_bonus_events.csv", index=False)
demerger_m.to_csv(INTERIM_DIR / "demerger_days.csv", index=False)
rights_ok.to_csv(INTERIM_DIR / "rights_events.csv", index=False)
big.to_csv(INTERIM_DIR / "big_price_moves.csv", index=False)

split/bonus events confirmed by the price drop               469  
split/bonus events moved to the matching drop                  0  
split/bonus events dropped (no price drop)                     3  
missing splits/bonuses added (price-implied)                   5  
large one-day moves kept as real (not reviewed)               15  
rights factors > 1.5 not applied (no price confirm.)           1  
manual demergers not mapped (MUST be 0)                        0  PASS


## Step 6: Monthly total returns

**Daily return** = (close × split factor + dividend) × rights factor ÷ yesterday's close − 1
- Yesterday's **actual** close is used (NSE `prevclose` is not adjusted). The dividend is per **old** share when it shares an ex-date with a split/bonus; on normal days the formula is (close + dividend) ÷ yesterday's close − 1.
- Demerger days: return = 0. Listing day: no return.
- **Months with no trades, and the month after one, have return NaN** (unknown, not 0).
- **Mergers into a listed stock:** the target's final month also earns (acquirer shares received × acquirer close on its next trading day) ÷ target's last close − 1. A swap return beyond ±25% means a wrong ratio (prices of merging companies stay aligned), so that merger is skipped and its returns stop at the last trade.
- Other delistings: returns stop at the last trade (no delisting return is available; a known limitation).

`ca_factor` = 1 ÷ split/bonus factor (per month); Step 9 uses it to carry share counts through splits and bonuses.

In [21]:
# ==========================================================
# STEP 6: Daily -> monthly total returns
# ==========================================================
d = daily.copy()

def attach(d, table, value_col, out_col, default):
    t = (table.dropna(subset=["trade_date"]).groupby(["sid", "trade_date"])[value_col].prod()
              .rename(out_col).reset_index().rename(columns={"trade_date": "date"}))
    d = d.merge(t, on=["sid", "date"], how="left")
    d[out_col] = d[out_col].fillna(default)
    return d

d = attach(d, split_m.drop_duplicates(["sid", "trade_date", "share_mult"]), "share_mult", "split_mult", 1.0)
d = attach(d, rights_ok, "ret_factor", "rights_mult", 1.0)
div_day = dividends_m.dropna(subset=["trade_date"]).groupby(["sid", "trade_date"])["div_ps"].sum()
d = d.merge(div_day.rename("div").reset_index().rename(columns={"trade_date": "date"}), on=["sid", "date"], how="left")
d["div"] = d["div"].fillna(0.0)
d["is_demerger"] = pd.Series(list(zip(d["sid"], d["date"]))).isin(set(zip(demerger_m["sid"], demerger_m["trade_date"]))).values

d["ret_d"] = (d["close"] * d["split_mult"] + d["div"]) * d["rights_mult"] / d["yesterday_close"] - 1
d.loc[d["is_demerger"], "ret_d"] = 0.0
d["log_ret"] = np.log1p(d["ret_d"])                          # monthly return = exp(sum of daily log returns) - 1
d["ca_factor"] = 1.0 / d["split_mult"]
d["is_event"] = (d["split_mult"] != 1) | (d["rights_mult"] != 1) | d["is_demerger"]
d["tt_day"] = d["series"].isin(["BE", "BZ"])
d["month"] = d["date"] + pd.offsets.MonthEnd(0)

g = d.groupby(["sid", "month"])
monthly = pd.DataFrame({
    "close":        g["close"].last(),                         # UNADJUSTED, for market cap only
    "traded_value": g["turnover"].sum(),                       # Rs traded in the month
    "trade_days":   g["close"].count(),
    "tt_days":      g["tt_day"].sum(),
    "ret":          np.expm1(g["log_ret"].sum()),
    "ca_factor":    g["ca_factor"].prod(),
    "dividends":    g["div"].sum(),
    "event_days":   g["is_event"].sum(),
    "last_trade":   g["date"].max(),
}).reset_index()


In [22]:
# Full month grid from first to last traded month (months with no trades get trade_days = 0)
grid = []
for sid, one in monthly.groupby("sid"):
    months = pd.date_range(one["month"].min(), one["month"].max(), freq=MONTH_END)
    one = one.set_index("month").reindex(months)
    one.index.name = "month"
    one["sid"] = sid
    grid.append(one.reset_index())
    
monthly = pd.concat(grid, ignore_index=True)
monthly["trade_days"] = monthly["trade_days"].fillna(0)
monthly["ca_factor"] = monthly["ca_factor"].fillna(1.0)

for c in ["traded_value", "tt_days", "dividends", "event_days"]:
    monthly[c] = monthly[c].fillna(0)
    
no_trade = (monthly["trade_days"] == 0) | (monthly.groupby("sid")["trade_days"].shift(1) == 0)
monthly.loc[no_trade, "ret"] = np.nan


In [23]:
# Mergers: final-month return includes the value of the acquirer shares received
monthly["merger_ret"] = np.nan
for target_sym, acq_sym, ratio in MERGERS:
    tgt = d[d["sid"] == resolve(target_sym, LAST_PRICE_DATE)]
    if tgt.empty:
        print("Merger skipped (target not in universe):", target_sym)
        continue
    
    acq = d[d["sid"] == resolve(acq_sym, tgt["date"].max())]      # acquirer's symbol at the merger date
    if acq.empty:
        print("Merger skipped (no prices):", target_sym, "->", acq_sym)
        continue
    
    last_day, last_close = tgt["date"].max(), tgt.loc[tgt["date"].idxmax(), "close"]
    nxt = acq[acq["date"] > last_day].head(1)
    
    if nxt.empty:
        continue
    swap = ratio * nxt["close"].iloc[0] / last_close - 1
    if abs(swap) > 0.25:                     # prices of merging companies stay aligned: a big gap = wrong ratio
        print(f"Merger {target_sym} -> {acq_sym}: SKIPPED, swap return {swap:.1%} implies a wrong ratio")
        continue
    
    row = (monthly["sid"] == tgt["sid"].iloc[0]) & (monthly["month"] == last_day + pd.offsets.MonthEnd(0))
    monthly.loc[row, "ret"] = (1 + monthly.loc[row, "ret"]) * (1 + swap) - 1
    monthly.loc[row, "merger_ret"] = swap
    print(f"Merger {target_sym} -> {acq_sym}: swap return {swap:.2%} added to {last_day:%b-%Y}")

price_data = monthly.rename(columns={"sid": "symbol", "month": "date"})
price_data = price_data[price_data["date"] <= pd.Timestamp(END_DATE)].sort_values(["symbol", "date"]).reset_index(drop=True)
print(price_data.shape, "| stocks:", price_data["symbol"].nunique(),
      "| months with an event:", int((price_data["event_days"] > 0).sum()))

Merger SBT -> SBIN: swap return 0.94% added to Mar-2017
Merger SBBJ -> SBIN: swap return 1.32% added to Mar-2017
Merger MYSOREBANK -> SBIN: swap return 1.39% added to Mar-2017
Merger ABIRLANUVO -> GRASIM: swap return 2.32% added to Jul-2017
Merger CAPF -> IDFCBANK: swap return 1.67% added to Dec-2018
Merger VIJAYABANK -> BANKBARODA: swap return -0.83% added to Mar-2019
Merger DENABANK -> BANKBARODA: swap return -1.22% added to Mar-2019
Merger BHARATFIN -> INDUSINDBK: swap return 4.65% added to Jul-2019
Merger GRUH -> BANDHANBNK: swap return 4.34% added to Oct-2019
Merger SYNDIBANK -> CANBK: swap return 0.38% added to Mar-2020
Merger ANDHRABANK -> UNIONBANK: swap return 3.79% added to Mar-2020
Merger CORPBANK -> UNIONBANK: swap return -1.84% added to Mar-2020
Merger ALBK -> INDIANB: swap return -20.71% added to Mar-2020
Merger ORIENTBANK -> PNB: swap return -6.31% added to Mar-2020
Merger UNITEDBNK -> PNB: swap return -4.81% added to Mar-2020
Merger GSKCONS -> HINDUNILVR: swap return -0

In [24]:
# ==========================================================
# STEP 6.1 CHECKS on returns
# ==========================================================
known = [("RELIANCE", "2017-09-30"), ("BAJFINANCE", "2016-09-30"), ("BAJFINANCE", "2025-06-30"),
         ("NYKAA", "2022-11-30"), ("JUBLFOOD", "2022-04-30"), ("VBL", "2023-06-30"),
         ("NESTLEIND", "2024-01-31"), ("JSWSTEEL", "2017-01-31"), ("ASTERDM", "2024-04-30"),
         ("HDFC", "2023-07-31")]

known_df = pd.DataFrame(known, columns=["symbol", "date"]).assign(date=lambda x: pd.to_datetime(x["date"]))
known_df["symbol"] = resolve_frame(known_df, "symbol", "date")

print("Check 1 - known split/bonus/dividend/merger months (should look normal, about -30% to +30%):")
display(known_df.merge(price_data[["symbol", "date", "ret", "dividends", "event_days", "merger_ret"]],
                       on=["symbol", "date"], how="left").round(3))

extreme = price_data[(price_data["ret"] < -0.5) | (price_data["ret"] > 1.0)]
print(f"\nCheck 2 - months with return < -50% or > +100%: {len(extreme)} (each should be a real move)")

display(extreme[["symbol", "date", "ret", "trade_days", "event_days", "dividends"]].sort_values("ret").round(3))

Check 1 - known split/bonus/dividend/merger months (should look normal, about -30% to +30%):


,symbol,date,ret,dividends,event_days,merger_ret
0,RELIANCE,2017-09-30,-0.021,0.0,1.0,NaN
1,BAJFINANCE,2016-09-30,-0.041,0.0,1.0,NaN
2,BAJFINANCE,2025-06-30,0.020,0.0,1.0,NaN
3,NYKAA,2022-11-30,-0.103,0.0,1.0,NaN
4,JUBLFOOD,2022-04-30,0.035,0.0,1.0,NaN
5,VBL,2023-06-30,-0.053,0.0,1.0,NaN
6,NESTLEIND,2024-01-31,-0.057,0.0,1.0,NaN
7,JSWSTEEL,2017-01-31,0.218,0.0,1.0,NaN
8,ASTERDM,2024-04-30,0.102,118.0,0.0,NaN
9,HDFC,2023-07-31,-0.023,0.0,0.0,0.012



Check 2 - months with return < -50% or > +100%: 247 (each should be a real move)


,symbol,date,ret,trade_days,event_days,dividends
75536,MINDACORP,2015-01-31,-0.892,21.0,0.0,0.0
34938,FEL,2016-05-31,-0.817,22.0,0.0,0.0
51964,ICIL,2016-11-30,-0.811,21.0,0.0,0.0
108805,SYMPHONY,2012-02-29,-0.777,20.0,0.0,0.0
66805,KNRCON,2016-12-31,-0.768,22.0,0.0,0.0
...,...,...,...,...,...,...
93818,RCOM,2017-12-31,1.869,20.0,0.0,0.0
83848,ORCHPHARMA,2020-12-31,1.900,22.0,0.0,0.0
84718,PATANJALI,2020-02-29,2.017,19.0,0.0,0.0
16834,BORORENEW,2020-06-30,2.800,22.0,0.0,0.0


## Step 7: Eligibility helpers and risk-free rate

**Eligibility helpers** (thresholds are set in Phase 2), each using only data up to that month: months with a trade so far, median monthly traded value over 12 months, trading days over 12 months.

**Risk-free rate:** RBI 91-day T-bill cut-off yield if `raw/rf/rbi_91d_tbill_cutoff.csv` exists; otherwise the FRED 3-month interbank rate for India (validated against IIMA's Rf in Phase 4: correlation 0.96, gap about 0.16% a year). Rf for month *t* = last month-end yield ÷ 12, i.e. known at the start of month *t* (FF rule). The FRED file is downloaded once and then reused.

In [25]:
# ==========================================================
# STEP 7.1: Eligibility helpers
# ==========================================================
price_data = price_data.sort_values(["symbol", "date"]).reset_index(drop=True)
g = price_data.groupby("symbol")
price_data["months_traded"] = g["trade_days"].transform(lambda s: (s > 0).cumsum())
price_data["tv_median_12m"] = g["traded_value"].transform(lambda s: s.rolling(12, min_periods=12).median())
price_data["trade_days_12m"] = g["trade_days"].transform(lambda s: s.rolling(12, min_periods=12).sum())
print(price_data[["months_traded", "tv_median_12m", "trade_days_12m"]].describe().round(1).to_string())

       months_traded  tv_median_12m  trade_days_12m
count       126202.0   1.165330e+05        116533.0
mean            84.4   1.163943e+10           246.0
std             54.0   2.960268e+10             7.6
min              1.0   0.000000e+00            21.0
25%             38.0   4.878031e+08           245.0
50%             79.0   2.145051e+09           247.0
75%            129.0   9.668716e+09           248.0
max            190.0   6.698988e+11           250.0


In [26]:
# ==========================================================
# STEP 7.2: Risk-free rate
# ==========================================================
RF_FILE = RF_DIR / "rbi_91d_tbill_cutoff.csv"
if RF_FILE.exists():
    rf = pd.read_csv(RF_FILE)
    rf["date"] = pd.to_datetime(rf["date"], dayfirst=True)
    RF_SOURCE = "RBI 91-day T-bill cut-off yield"
else:
    fred_files = sorted(RF_DIR.glob("fred_INDIR3TIB01STM_*.csv"))
    newest_ok = fred_files and date.fromisoformat(fred_files[-1].stem[-10:]) > END_DATE
    if newest_ok:
        fred = pd.read_csv(fred_files[-1])
    else:
        try:
            fred = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=INDIR3TIB01STM")
            fred.to_csv(RF_DIR / f"fred_INDIR3TIB01STM_{RUN_DATE.isoformat()}.csv", index=False)
        except Exception as e:
            print("FRED download failed, using the newest saved copy:", type(e).__name__)
            fred = pd.read_csv(fred_files[-1])
    rf = fred.rename(columns={"observation_date": "date", "INDIR3TIB01STM": "yield_pct"})
    rf["date"] = pd.to_datetime(rf["date"])
    RF_SOURCE = "FRED INDIR3TIB01STM (3-month interbank rate, proxy for the 91-day T-bill)"

rf["yield_pct"] = pd.to_numeric(rf["yield_pct"], errors="coerce")
rf = rf.dropna(subset=["yield_pct"]).sort_values("date").set_index("date")
rf_monthly = (rf["yield_pct"].resample(MONTH_END).last().ffill() / 100 / 12).shift(1)
rf_monthly = rf_monthly.rename("Rf_monthly").to_frame()
rf_monthly.index.name = "date"
rf_monthly.to_csv(PROCESSED_DIR / "rf_monthly.csv")

have_rf = set(rf_monthly.dropna().index)
no_rf = sorted(set(price_data["date"]) - have_rf)
print("Rf source:", RF_SOURCE)
print("Rf:", min(have_rf).date(), "->", max(have_rf).date(), "| return months without Rf:", len(no_rf),
      "| latest ones:", [m.date() for m in no_rf][-3:])

Rf source: FRED INDIR3TIB01STM (3-month interbank rate, proxy for the 91-day T-bill)
Rf: 2011-12-31 -> 2026-09-30 | return months without Rf: 12 | latest ones: [datetime.date(2011, 10, 31), datetime.date(2011, 11, 30), datetime.date(2026, 10, 31)]


## Step 8: Fundamentals (Screener.in)

Annual P&L and balance sheet for every stock that was ever in the universe, **including delisted and merged ones** (Screener keeps their pages, e.g. HDFC Ltd). Pages are cached in `raw/screener_html/<snapshot>/`; only missing pages are downloaded.

- **Basis:** consolidated where available, standalone fills missing years (Compustat convention). A year without a balance sheet is dropped.
- **Sector:** Screener's "Broad Sector" (NSE classification). **`is_financial` = Broad Sector "Financial Services"**; Phase 2 excludes these from the RMW sort only.
- **Face value** (latest) and **equity capital** (each year) are kept: they give a share-count estimate for stocks with no other share data (Step 9).
- Units: ₹ crore, as on Screener. A stock with no Screener data stays in MKT and the size sort but not in the B/M, OP and investment sorts (FF rule).

In [27]:
# ==========================================================
# STEP 8: Screener fundamentals for every universe stock
# ==========================================================

def create_session():
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=2,
                                                      status_forcelist=[429, 500, 502, 503, 504])))
    s.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
                      "Accept": "text/html,application/xhtml+xml"})
    return s


def get_html_cached(url, symbol, session, min_len=5000):
    kind = "consolidated" if "/consolidated/" in url else "standalone"
    path = SCREENER_DIR / f"{symbol}_{kind}.html"
    if path.exists():
        return path.read_text(encoding="utf-8")
    if session is None:
        return None
    res = session.get(url, timeout=15)
    if res.status_code != 200 or len(res.text) < min_len or "No data" in res.text:
        return None                                   # failures are not cached, so they retry next run
    path.write_text(res.text, encoding="utf-8")
    time.sleep(1.5)
    return res.text


EMPTY = {"", "-", "--", "na", "n/a", "nan"}

def to_float(text):
    text = str(text).replace(",", "").replace("%", "").replace("₹", "").strip()
    if text.lower() in EMPTY:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def read_screener_table(table):
    """Screener table -> {fiscal_year_end: {row_label: value}} (skips 'TTM' and half-year columns)."""
    rows = table.find_all("tr")
    headers = [c.get_text(strip=True) for c in rows[0].find_all(["th", "td"])]
    data = {}
    for row in rows[1:]:
        cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
        if len(cells) < 2:
            continue
        label = cells[0].replace("\xa0", " ").rstrip("+").strip().lower()
        for header, value in zip(headers[1:], cells[1:]):
            m = re.search(r"([A-Za-z]{3})\s*(\d{4})", header)
            if m is None:
                continue
            fy_end = pd.to_datetime(f"1 {m.group(1)} {m.group(2)}") + pd.offsets.MonthEnd(0)
            year_dict = data.setdefault(fy_end, {})
            if year_dict.get(label) is None:
                year_dict[label] = to_float(value)
    return data


def pick(values, names):
    for name in names:
        if values.get(name) is not None:
            return values[name]
    return None


def parse_screener_page(html):
    """One Screener page -> (annual DataFrame in Rs crore, meta dict with sector and face value)."""
    soup = BeautifulSoup(html, "html.parser")

    meta = {"broad_sector": None, "industry": None, "face_value": None}
    peers = soup.find(id="peers")
    if peers is not None:
        for a in peers.find_all("a"):
            if a.get("title") == "Broad Sector":
                meta["broad_sector"] = a.get_text(strip=True)
            elif a.get("title") == "Industry":
                meta["industry"] = a.get_text(strip=True)
    ratios = soup.find(id="top-ratios")
    if ratios is not None:
        for li in ratios.find_all("li"):
            spans = li.find_all("span")
            if spans and spans[0].get_text(strip=True).lower() == "face value":
                meta["face_value"] = to_float(li.find(class_="number").get_text(strip=True)
                                              if li.find(class_="number") else spans[-1].get_text(strip=True))

    pnl_table = bal_table = None
    for sec in soup.find_all("section"):
        heading, tbl = sec.find(["h2", "h3"]), sec.find("table")
        if heading is None or tbl is None:
            continue
        title = heading.get_text(strip=True).lower()
        if title == "profit & loss" and pnl_table is None:
            pnl_table = tbl
        elif title == "balance sheet" and bal_table is None:
            bal_table = tbl
    if pnl_table is None or bal_table is None:
        return None, meta

    pnl, bal = read_screener_table(pnl_table), read_screener_table(bal_table)
    records = []
    for fy_end in sorted(pnl):
        p, b = pnl[fy_end], bal.get(fy_end, {})        # SAME fiscal year only
        cap, res = pick(b, ["equity capital", "share capital"]), pick(b, ["reserves"])
        records.append({
            "fy_end": fy_end, "year": fy_end.year,
            "sales": pick(p, ["sales", "revenue"]),
            "operating_profit": pick(p, ["operating profit"]),     # before depreciation (EBITDA)
            "other_income": pick(p, ["other income"]),
            "interest_expense": pick(p, ["interest"]),
            "depreciation": pick(p, ["depreciation"]),
            "net_income": pick(p, ["net profit"]),
            "total_assets": pick(b, ["total assets"]),
            "equity_capital": cap,
            "book_value": cap + res if (cap is not None and res is not None) else None,
        })
    return pd.DataFrame(records), meta


def run_pipeline(symbol, session):
    frames, meta = [], {}
    for basis, url in [("consolidated", f"https://www.screener.in/company/{symbol}/consolidated/?view=15years"),
                       ("standalone",   f"https://www.screener.in/company/{symbol}/?view=15years")]:
        try:
            html = get_html_cached(url, symbol, session)
        except Exception as e:
            print(f"  [{symbol} {basis}] network error: {type(e).__name__}")
            continue
        if html is None:
            continue
        df, m = parse_screener_page(html)
        meta = {k: v for k, v in {**m, **meta}.items() if v is not None} or meta
        if df is not None and not df.empty:
            frames.append(df.assign(basis=basis))
    if not frames:
        return None, meta
    both = pd.concat(frames, ignore_index=True)
    both = both[both["total_assets"].notna()]
    both["basis_rank"] = both["basis"].map({"consolidated": 0, "standalone": 1})
    both = (both.sort_values(["fy_end", "basis_rank"]).drop_duplicates("fy_end", keep="first")
                .drop(columns="basis_rank"))
    both.insert(0, "symbol", symbol)
    return both.sort_values("fy_end").reset_index(drop=True), meta


In [28]:
try:
    session = create_session()
except Exception:
    session = None                                  # offline: cached pages only

all_fund, metas, failures = [], [], []
for i, sym in enumerate(sorted(UNIVERSE_SIDS), 1):
    df, meta = run_pipeline(sym, session)
    metas.append({"symbol": sym, **meta})
    if df is None or df.empty:
        failures.append(sym)
    else:
        all_fund.append(df)
    if i % 100 == 0:
        print(f"  {i}/{len(UNIVERSE_SIDS)} done")

final_df = pd.concat(all_fund, ignore_index=True)

# Screener blanks arrive as None -> columns become 'object'; make every accounting column numeric
NUM_COLS = ["sales", "operating_profit", "other_income", "interest_expense", "depreciation",
            "net_income", "total_assets", "equity_capital", "book_value"]
final_df[NUM_COLS] = final_df[NUM_COLS].apply(pd.to_numeric, errors="coerce")

meta_df = pd.DataFrame(metas)

# Sector / financial flag: Screener first, NSE's current Nifty 500 list as fallback
nse_list_file = UNIVERSE_DIR / f"nifty500_{RUN_DATE.isoformat()}.csv"
if not list(UNIVERSE_DIR.glob("nifty500_*.csv")):
    try:
        pd.read_csv(NIFTY500_LIST_URL).to_csv(nse_list_file, index=False)
    except Exception:
        pass
nse_lists = sorted(UNIVERSE_DIR.glob("nifty500_*.csv"))
if nse_lists:
    nse_ind = pd.read_csv(nse_lists[-1])[["Symbol", "Industry"]].rename(columns={"Symbol": "symbol"})
    meta_df = meta_df.merge(nse_ind, on="symbol", how="left")
    meta_df["broad_sector"] = meta_df["broad_sector"].fillna(meta_df["Industry"])
    meta_df = meta_df.drop(columns="Industry")
meta_df["is_financial"] = meta_df["broad_sector"].eq("Financial Services")

# FF5 operating profitability: (operating profit - interest) / book equity, same fiscal year
ok = final_df["operating_profit"].notna() & final_df["interest_expense"].notna() & (final_df["book_value"] > 0)

final_df["operating_profitability"] = (
    (final_df["operating_profit"] - final_df["interest_expense"]) / final_df["book_value"]).where(ok)
final_df = final_df.merge(meta_df[["symbol", "is_financial", "broad_sector"]], on="symbol", how="left")
final_df["months_since_prev_fy"] = (final_df.groupby("symbol")["fy_end"].diff().dt.days / 30.44).round()
final_df["prev_basis"] = final_df.groupby("symbol")["basis"].shift(1)

# Data-error flag: tiny book equity vs assets (column name kept for Phase 2 compatibility)
final_df["check_me"] = (final_df["book_value"] > 0) & (final_df["book_value"] < 0.01 * final_df["total_assets"])
final_df["available_date"] = final_df["fy_end"] + pd.offsets.MonthEnd(LAG_MONTHS)   # SEBI LODR: 60 days
final_df.to_csv(PROCESSED_DIR / "final_fin_data.csv", index=False)

print(f"Stocks with fundamentals: {final_df['symbol'].nunique()} of {len(UNIVERSE_SIDS)} "
      f"| without: {len(failures)} | sector known: {meta_df['broad_sector'].notna().sum()}")
print("FY range:", final_df["fy_end"].min().date(), "->", final_df["fy_end"].max().date(),
      "| financial firms:", int(meta_df["is_financial"].sum()))
print("Stocks without Screener data (first 20):", failures[:20])

C:\Users\jayan\AppData\Local\Temp\ipykernel_39496\2968719731.py:142: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  both = pd.concat(frames, ignore_index=True)
C:\Users\jayan\AppData\Local\Temp\ipykernel_39496\2968719731.py:142: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  both = pd.concat(frames, ignore_index=True)


  100/879 done
  200/879 done
  300/879 done


C:\Users\jayan\AppData\Local\Temp\ipykernel_39496\2968719731.py:142: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  both = pd.concat(frames, ignore_index=True)


  400/879 done
  500/879 done
  600/879 done
  700/879 done
  800/879 done


C:\Users\jayan\AppData\Local\Temp\ipykernel_39496\840949299.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(all_fund, ignore_index=True)


Stocks with fundamentals: 844 of 879 | without: 35 | sector known: 823
FY range: 2002-03-31 -> 2026-06-30 | financial firms: 128
Stocks without Screener data (first 20): ['COX&KINGS', 'DALMIABHA', 'DHANI', 'DHFL', 'EQUITAS', 'FSC', 'GALLISPAT', 'GRUH', 'GSKCONS', 'GSPL', 'HDFC', 'HEXAWARE', 'IDFC', 'INOXLEISUR', 'ISEC', 'IWEL', 'JBCHEPHARM', 'JPASSOCIAT', 'JSLHISAR', 'MEGH']


## Step 9: Share counts and market cap (ME)

ME = **unadjusted close × total shares** (FF rule: total shares, not free float), in ₹ crore.

Share counts come from three sources. In every month the first available one is used:

| Priority | Source | Coverage |
|---|---|---|
| 1 | **yfinance** reported share count (raw, not split-adjusted; monthly median; not used in a month with a split/bonus) | current stocks, ~2017 onwards |
| 2 | **NSE market cap ÷ close** on the ranking file's date (6-month average files: average market cap ÷ average split-adjusted close) | every ranked stock, 2019 onwards |
| 3 | **Screener equity capital ÷ face value** at each fiscal year-end (face value moved back through splits) | every stock with a Screener page, incl. delisted |

Between those points the count is carried **in split-free units** (count × running product of 1/split-bonus factor), which stay constant through splits and bonuses, so gaps are filled forward and backward safely. 

**Consistency rules** (compared in split-free units): Screener counts that disagree with NSE's by more than 10% (median) are not used for that stock; **each monthly yfinance count is used only if it is within 25% of the NSE/Screener anchor level** (this removes one-month glitches and early yfinance history that already includes later bonuses/splits).

**Limitation:** share issues/buybacks between two data points are not captured; Screener rounds equity capital to whole ₹ crore.

In [29]:
# ==========================================================
# STEP 9.1: yfinance share counts (cached; one value per month = median of reported counts)
# ==========================================================

def get_shares(symbol):
    path = SHARES_DIR / f"{symbol}.csv"
    if path.exists():
        return pd.read_csv(path, parse_dates=["date"])
    try:
        import yfinance as yf
        s = yf.Ticker(f"{symbol}.NS").get_shares_full(start=str(START_DATE))
    except Exception:
        return None
    if s is None or len(s) == 0:
        pd.DataFrame(columns=["date", "shares"]).to_csv(path, index=False)   # remember "no data"
        return None
    dates = pd.to_datetime(s.index)
    dates = dates.tz_localize(None) if dates.tz is not None else dates
    df = pd.DataFrame({"date": dates, "shares": s.values})
    df.to_csv(path, index=False)
    time.sleep(0.5)
    return df


yf_rows = []
for sym in sorted(UNIVERSE_SIDS):
    one = get_shares(sym)
    if one is None or one.empty:
        continue
    m = one.set_index("date")["shares"].resample(MONTH_END).median().dropna()
    yf_rows.append(pd.DataFrame({"symbol": sym, "date": m.index, "shares_yf": m.values}))
shares_yf = pd.concat(yf_rows, ignore_index=True) if yf_rows else pd.DataFrame(columns=["symbol", "date", "shares_yf"])
print("Stocks with yfinance share counts:", shares_yf["symbol"].nunique())

Stocks with yfinance share counts: 856


In [30]:
# ==========================================================
# STEP 9.2: NSE anchors (market cap / close) and Screener anchors (equity capital / face value)
# ==========================================================

# --- NSE anchors ---
nse_anchor_rows = []
for (as_of, kind), r in rankings[rankings["mcap_cr"].notna() & rankings["sid"].isin(UNIVERSE_SIDS)].groupby(["as_of", "kind"]):
    if kind == "average":
        start = as_of - pd.DateOffset(months=6) + pd.Timedelta(days=1)
        win = daily[(daily["date"] >= start) & (daily["date"] <= as_of) & daily["sid"].isin(set(r["sid"]))].copy()
        # Express every price in end-of-period share units: divide by splits/bonuses after that day
        ev_in = split_m[(split_m["trade_date"] > start) & (split_m["trade_date"] <= as_of)]
        win["adj"] = 1.0
        for sid, t, mult in zip(ev_in["sid"], ev_in["trade_date"], ev_in["share_mult"]):
            win.loc[(win["sid"] == sid) & (win["date"] < t), "adj"] *= mult
        px = (win["close"] / win["adj"]).groupby(win["sid"]).mean()
    else:
        px = daily[(daily["date"] <= as_of) & (daily["date"] > as_of - pd.Timedelta(days=10))] \
                .sort_values("date").groupby("sid")["close"].last()
    a = r.set_index("sid")["mcap_cr"].to_frame().join(px.rename("px"), how="inner")
    a["shares_nse"] = a["mcap_cr"] * 1e7 / a["px"]
    a["date"] = as_of + pd.offsets.MonthEnd(0)
    nse_anchor_rows.append(a.reset_index()[["sid", "date", "shares_nse"]])
nse_anchor = pd.concat(nse_anchor_rows, ignore_index=True).rename(columns={"sid": "symbol"})


In [31]:
# --- Screener anchors: shares = equity capital / face value at that fiscal year-end ---
# Face value at date t = latest face value x product of split factors AFTER t
fv_now = meta_df.set_index("symbol")["face_value"]
splits_only = split_m[(split_m["fv_mult"] != 1) & split_m["trade_date"].notna()]
scr = final_df[["symbol", "fy_end", "equity_capital"]].dropna().copy()
scr["fv"] = scr["symbol"].map(fv_now)
scr["fv_factor"] = [
    splits_only.loc[(splits_only["sid"] == s) & (splits_only["trade_date"] > t), "fv_mult"].prod()
    for s, t in zip(scr["symbol"], scr["fy_end"])]
scr["shares_scr"] = scr["equity_capital"] * 1e7 / (scr["fv"] * scr["fv_factor"])
scr_anchor = scr.dropna(subset=["shares_scr"]).rename(columns={"fy_end": "date"})[["symbol", "date", "shares_scr"]]

print("NSE anchors:", len(nse_anchor), "| Screener anchors:", len(scr_anchor))

NSE anchors: 8816 | Screener anchors: 9764


In [32]:
# ==========================================================
# STEP 9.3: Combine sources -> monthly shares -> ME
# ==========================================================

pdx = price_data.merge(shares_yf, on=["symbol", "date"], how="left") \
                .merge(nse_anchor, on=["symbol", "date"], how="left") \
                .merge(scr_anchor, on=["symbol", "date"], how="left")

pdx = pdx.sort_values(["symbol", "date"]).reset_index(drop=True)

# In a month with a split/bonus, yfinance's monthly median mixes counts from before and after the event:
# ignore it there and let the split-free carry (below) handle the event
pdx.loc[pdx["ca_factor"] != 1, "shares_yf"] = np.nan

# ---- Consistency rules (all comparisons in split-free units, so splits/bonuses are not disagreements) ----
pdx["split_index"] = pdx.groupby("symbol")["ca_factor"].cumprod()

# Rule 1 (per stock): Screener counts that disagree with NSE by > 10% (median) are not used for that stock
def median_gap(a, b):
    left = pdx[["symbol", "date"]].assign(a=pdx[a] * pdx["split_index"]).dropna().sort_values("date")
    right = pdx[["symbol", "date"]].assign(b=pdx[b] * pdx["split_index"]).dropna().sort_values("date")
    m = pd.merge_asof(left, right, on="date", by="symbol", direction="nearest",
                      tolerance=pd.Timedelta(days=366)).dropna()
    return (m["a"] / m["b"] - 1).abs().groupby(m["symbol"]).median()

gap_scr = median_gap("shares_scr", "shares_nse")
bad_scr = set(gap_scr[gap_scr > 0.10].index)
pdx.loc[pdx["symbol"].isin(bad_scr), "shares_scr"] = np.nan

# Rule 2 (per month): a yfinance count is used only if it is within 25% of the anchor level
# (NSE, else Screener, carried in split-free units). This removes one-month glitches and early
# yfinance history that already includes later bonuses/splits. With no anchor, yfinance is kept.
anchor_norm = (pdx["shares_nse"].fillna(pdx["shares_scr"]) * pdx["split_index"])
anchor_norm = anchor_norm.groupby(pdx["symbol"]).transform(lambda s: s.ffill().bfill())

yf_norm = pdx["shares_yf"] * pdx["split_index"]
yf_bad = yf_norm.notna() & anchor_norm.notna() & ((yf_norm / anchor_norm - 1).abs() > 0.25)

pdx.loc[yf_bad, "shares_yf"] = np.nan
pdx = pdx.drop(columns="split_index")

print(f"Screener vs NSE: {len(gap_scr)} stocks compared | median gap {gap_scr.median():.2%} "
      f"| Screener dropped for {len(bad_scr)}: {sorted(bad_scr)[:10]}")
print(f"yfinance months rejected (> 25% from the NSE/Screener anchor): {int(yf_bad.sum())} "
      f"in {pdx.loc[yf_bad, 'symbol'].nunique()} stocks")

def fill_shares(one):
    one = one.copy()
    one["split_index"] = one["ca_factor"].cumprod()               # halves at a 1:1 bonus
    raw = one["shares_yf"].fillna(one["shares_nse"]).fillna(one["shares_scr"])
    one["shares_source"] = np.select([one["shares_yf"].notna(), one["shares_nse"].notna(),
                                      one["shares_scr"].notna()], ["yfinance", "nse", "screener"], default="")
    norm = raw * one["split_index"]                               # split-free units
    one["shares"] = norm.ffill().bfill() / one["split_index"]
    one.loc[(one["shares_source"] == "") & norm.ffill().notna(), "shares_source"] = "carried forward"
    one.loc[(one["shares_source"] == "") & one["shares"].notna(), "shares_source"] = "backfilled"
    one.loc[one["shares"].isna(), "shares_source"] = "missing"
    return one

price_data = pd.concat([fill_shares(one) for _, one in pdx.groupby("symbol")], ignore_index=True)
price_data["me"] = price_data["close"] * price_data["shares"] / 1e7          # Rs crore
price_data["me_lag"] = price_data.groupby("symbol")["me"].shift(1)           # weight for this month's return
price_data.to_csv(PROCESSED_DIR / "monthly_prices.csv", index=False)

print(price_data["shares_source"].value_counts().to_string())

Screener vs NSE: 824 stocks compared | median gap 0.37% | Screener dropped for 10: ['ALBK', 'CCAVENUE', 'GENUSPOWER', 'HMT', 'JSWSTEEL', 'M&M', 'TASTYBITE', 'TMPV', 'UNITEDBNK', 'VEEDOL']
yfinance months rejected (> 25% from the NSE/Screener anchor): 675 in 126 stocks
shares_source
yfinance           66664
carried forward    28503
backfilled         26652
screener            2688
nse                 1520
missing              175


## Step 10: Characteristics at each June formation, and the monthly panel

**Definitions (FF5, Indian data)**
- **Size** = ME at the end of June *t*.
- **B/M** = book equity (equity capital + reserves) ÷ ME, both at the firm's own fiscal year-end (India's version of FF's December ME). Negative book equity → B/M missing.
- **OP** = (operating profit before depreciation − interest) ÷ book equity, same fiscal year.
- **Investment** = total assets FY *t* ÷ total assets FY *t*−1 − 1 (only if 12 months apart and on the same basis).
- **One fiscal year per stock per formation:** the latest fiscal year public by June *t* (3-month lag, SEBI LODR Reg 33), ending within the 12 months before that.
- **Characteristics rows = universe members** (all of them, with or without fundamentals). The **panel** has every month of every stock ever in the universe, flagged `in_universe` for the holding year (July *t* – June *t*+1) of each June it was a member.

In [33]:
# ==========================================================
# STEP 10.1: Fundamentals per fiscal year (asset growth, ME at FY end, B/M)
# ==========================================================
fund = final_df.sort_values(["symbol", "fy_end"]).reset_index(drop=True)
fund["prev_total_assets"] = fund.groupby("symbol")["total_assets"].shift(1)
fund["asset_growth"] = fund["total_assets"] / fund["prev_total_assets"] - 1
bad = (fund["months_since_prev_fy"] != 12) | (fund["basis"] != fund["prev_basis"]) | (fund["prev_total_assets"] <= 0)
fund.loc[bad, "asset_growth"] = np.nan

me_fy = price_data[["symbol", "date", "me"]].rename(columns={"date": "fy_end", "me": "me_fy_end"})
fund = fund.merge(me_fy, on=["symbol", "fy_end"], how="left", validate="1:1")
fund["bm"] = fund["book_value"] / fund["me_fy_end"]
fund.loc[(fund["book_value"] <= 0) | (fund["me_fy_end"] <= 0), "bm"] = np.nan


def fundamentals_as_of(fund, formation_date):
    """Latest fiscal year PUBLIC on formation_date, ending in the 12 months before (formation_date - lag)."""
    F = pd.Timestamp(formation_date)
    oldest_ok = F - pd.offsets.MonthEnd(LAG_MONTHS + 12)
    usable = fund[(fund["available_date"] <= F) & (fund["fy_end"] > oldest_ok)]
    return usable.sort_values("fy_end").groupby("symbol").tail(1)


# Test: a June-2017 formation must pick FY Mar-2017, never Mar-2016
print(fundamentals_as_of(fund, "2017-06-30")["fy_end"].value_counts().head(3).to_string())

fy_end
2017-03-31    722
2016-12-31     18
2016-06-30      3


In [34]:
# ==========================================================
# STEP 10.2: Characteristics for every universe member at each June
# ==========================================================
snaps = []
for F, members in universe.groupby("formation_date"):
    snap = members[["formation_date", "sid", "rank"]].rename(columns={"sid": "symbol", "rank": "nse_rank"})
    snap = snap.merge(fundamentals_as_of(fund, F).drop(columns=["is_financial", "broad_sector"]),
                      on="symbol", how="left")
    snaps.append(snap)
chars = pd.concat(snaps, ignore_index=True)
chars = chars.merge(meta_df[["symbol", "is_financial", "broad_sector"]], on="symbol", how="left")
chars["is_financial"] = chars["is_financial"].fillna(False).astype(bool)

at_F = price_data[["symbol", "date", "me", "shares_source", "months_traded", "tv_median_12m", "trade_days_12m"]]
at_F = at_F.rename(columns={"date": "formation_date", "me": "me_formation"})
chars = chars.merge(at_F, on=["symbol", "formation_date"], how="left", validate="1:1")
chars["in_universe"] = True

In [35]:
# ==========================================================
# STEP 10.3: Monthly panel
# ==========================================================
panel = price_data[["symbol", "date", "ret", "close", "shares", "me", "me_lag",
                    "shares_source", "trade_days", "tt_days"]].copy()
panel = panel.merge(rf_monthly.reset_index(), on="date", how="left", validate="m:1")
panel["excess_ret"] = panel["ret"] - panel["Rf_monthly"]


def formation_date_for(month_end):
    """Jul-2017 ... Jun-2018 -> 2017-06-30 (the June that formed the portfolio)."""
    year = month_end.year if month_end.month > FORMATION_MONTH else month_end.year - 1
    return pd.Timestamp(year, FORMATION_MONTH, 1) + pd.offsets.MonthEnd(0)


panel["formation_date"] = panel["date"].map(formation_date_for)
members = set(zip(universe["sid"], universe["formation_date"]))
panel["in_universe"] = [(s, f) in members for s, f in zip(panel["symbol"], panel["formation_date"])]
panel = panel.merge(meta_df[["symbol", "is_financial", "broad_sector"]], on="symbol", how="left")
panel["is_financial"] = panel["is_financial"].fillna(False).astype(bool)
print("Panel:", panel.shape, "| universe stock-months:", int(panel["in_universe"].sum()))

Panel: (126202, 16) | universe stock-months: 61677


In [36]:
# ==========================================================
# STEP 10.4: Nifty total-return indices (for Step 11 checks 6-7 and Phase 4)
# niftyindices.com serves at most 1 year per request; later runs fetch only new days
# ==========================================================
import requests

TRI_INDICES = {"nifty500": "NIFTY 500", "nifty100": "NIFTY 100",
               "midcap150": "NIFTY MIDCAP 150", "smallcap250": "NIFTY SMALLCAP 250"}
TRI_FIRST_DAY = pd.Timestamp("2016-06-01")          # so the July 2016 return can be computed

TRI_URLS = ["https://niftyindices.com/Backpage.aspx/getTotalReturnIndexString",
            "https://www.niftyindices.com/Backpage.aspx/getTotalReturnIndexString",
            "https://www.niftyindices.com/Backpage/getTotalReturnIndexString"]
TRI_HEADERS = {
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Content-Type": "application/json; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Origin": "https://niftyindices.com",
    "Referer": "https://niftyindices.com/reports/historical-data",
}


def download_tri(session, index_name, start, end):
    """Daily TRI for one index, start/end as 'DD-Mon-YYYY' (max 1 year apart)."""
    cinfo = (f"{{'name':'{index_name}','startDate':'{start}',"
             f"'endDate':'{end}','indexName':'{index_name}'}}")
    for url in TRI_URLS:
        r = session.post(url, headers=TRI_HEADERS, json={"cinfo": cinfo}, timeout=30)
        if r.status_code == 200:
            data = r.json()
            if isinstance(data, dict) and "d" in data:      # old site: rows inside "d" as text
                data = json.loads(data["d"])
            return pd.DataFrame(data)
    raise RuntimeError(f"all TRI URLs failed for {index_name} {start}")


try:
    tri_session = requests.Session()
    tri_session.get("https://niftyindices.com/reports/historical-data",
                    headers={"User-Agent": TRI_HEADERS["User-Agent"]}, timeout=30)   # site cookies
except Exception as e:
    tri_session = None
    print("niftyindices.com not reachable, saved TRI files are used:", type(e).__name__)

for short_name, index_name in TRI_INDICES.items():
    path = TRI_DIR / f"{short_name}.csv"
    tri = pd.read_csv(path) if path.exists() else pd.DataFrame()
    last_day = parse_dates(tri["Date"]).max() if len(tri) else TRI_FIRST_DAY - pd.Timedelta(days=1)

    start, pieces = last_day + pd.Timedelta(days=1), []
    while tri_session is not None and start <= pd.Timestamp(END_DATE):
        end = min(start + pd.DateOffset(years=1) - pd.Timedelta(days=1), pd.Timestamp(END_DATE))
        try:
            pieces.append(download_tri(tri_session, index_name, f"{start:%d-%b-%Y}", f"{end:%d-%b-%Y}"))
        except Exception as e:
            print(f"  {short_name}: download stopped at {start.date()} ({type(e).__name__})")
            break
        start = end + pd.Timedelta(days=1)
        time.sleep(1)

    if pieces:
        tri = pd.concat([tri] + pieces, ignore_index=True).drop_duplicates(subset="Date")
        tri.to_csv(path, index=False)
    last = parse_dates(tri["Date"]).max().date() if len(tri) else "no file"
    print(f"{short_name:12s} rows: {len(tri):5d} | last day: {last}")

  nifty500: download stopped at 2026-07-01 (JSONDecodeError)
nifty500     rows:  2497 | last day: 2026-06-30
  nifty100: download stopped at 2026-07-01 (JSONDecodeError)
nifty100     rows:  2497 | last day: 2026-06-30
  midcap150: download stopped at 2026-07-01 (JSONDecodeError)
midcap150    rows:  2497 | last day: 2026-06-30
  smallcap250: download stopped at 2026-07-01 (JSONDecodeError)
smallcap250  rows:  2497 | last day: 2026-06-30


## Step 11: Final checks

1. Universe: 500 members each June; members without prices, ME or fundamentals.
2. Share-count sources by formation year.
3. Share counts that jump > 50% in a month without a split/bonus (should be rare: real issues, mergers, or source switches).
4. Size sanity: the 10 largest stocks each June must be the familiar large caps.
5. Members that delisted (last trade before the cutoff): mergers into a listed stock belong in `MERGERS` (Step 5.3). **5b** lists likely renames missing from NSE's symbol-change list (a new symbol starting within 10 days at about the same price); confirm them and add them to `MANUAL_RENAMES` (Step 1).
6. **Market:** value-weighted return of universe members vs the Nifty 500 Total Return Index (survivorship-free). Expect correlation ≥ 0.98 and a small CAGR gap (remaining differences: full vs free-float weights, eligibility rules).
7. **Size:** small-half minus big-half return of universe members vs the survivorship-free index spread (Midcap 150 + Smallcap 250)/2 − Nifty 100. With survivorship removed, the two averages should be close.

Checks 6–7 need the NSE TRI files in `raw/external/nse_tri/` (downloaded in Step 10.4).

In [37]:
# ==========================================================
# STEP 11: Final checks
# ==========================================================
# 1) Universe coverage
cov = chars.groupby("formation_date").agg(
    members=("symbol", "count"), with_me=("me_formation", "count"), with_bm=("bm", "count"),
    with_op=("operating_profitability", "count"), with_inv=("asset_growth", "count"),
    financials=("is_financial", "sum"))
print("1) Universe coverage per June:"); print(cov.to_string())

# 2) Share-count sources
print("\n2) Share-count source at formation:")
print(chars.groupby("formation_date")["shares_source"].value_counts().unstack().fillna(0).astype(int).to_string())

# 3) Suspicious share jumps
price_data["shares_change"] = price_data.groupby("symbol")["shares"].pct_change(fill_method=None)
jumps = price_data[(price_data["shares_change"].abs() > 0.5) & (price_data["ca_factor"] == 1)]
print("\n3) Share counts jumping > 50% without a split/bonus:", len(jumps))
display(jumps[["symbol", "date", "shares", "shares_change", "shares_source"]].head(15))

# 4) Size sanity
print("\n4) Largest stocks by June ME:")
for F in sorted(chars["formation_date"].unique())[::3]:
    top = chars[chars["formation_date"] == F].nlargest(10, "me_formation")
    print(pd.Timestamp(F).date(), ", ".join(f"{s} ({m:,.0f})" for s, m in zip(top["symbol"], top["me_formation"])))

# 5) Delisted members
last = price_data.groupby("symbol")["last_trade"].max()
gone = last[last < LAST_PRICE_DATE - pd.Timedelta(days=31)]
merged = {resolve(t, LAST_PRICE_DATE) for t, _, _ in MERGERS}
print(f"\n5) Universe stocks that stopped trading: {len(gone)} | handled in MERGERS: {len(merged & set(gone.index))}")
print(gone.sort_values().dt.date.to_string())

# 5b) Possible renames missing from NSE's list: a symbol whose FIRST trade is within 10 days after a stopped
#     stock's last trade, at a price within 10% of that stock's last close. Confirm on Screener/NSE, then add
#     the printed line to MANUAL_RENAMES in Step 1 (it is already in the right format).
first = (bhav.sort_values("date").groupby("symbol", observed=True)
             .agg(first_date=("date", "first"), first_close=("close", "first")).reset_index())
first["symbol"] = first["symbol"].astype(str)
first["first_close"] = pd.to_numeric(first["first_close"], errors="coerce")
last_close = daily.sort_values("date").groupby("sid")["close"].last()
print("\n5b) Possible renames (old, new, first day under new symbol, price ratio):")
n_found = 0

for sid, L in gone.items():
    cand = first[(first["first_date"] > L) & (first["first_date"] <= L + pd.Timedelta(days=10))]
    cand = cand[(cand["first_close"] / last_close[sid] - 1).abs() <= 0.10]
    for _, c in cand.iterrows():
        n_found += 1
        print(f'    ("{sid}", "{c["symbol"]}", "{c["first_date"]:%Y-%m-%d}"),   # price ratio '
              f'{c["first_close"] / last_close[sid]:.3f}')

print("  candidates found:", n_found)

# 6) and 7) Market and size vs survivorship-free NSE indices
def tri_returns(name):
    t = pd.read_csv(TRI_DIR / f"{name}.csv")
    t["date"] = parse_dates(t["Date"])
    t["tri"] = to_number(t["TotalReturnsIndex"])
    return t.dropna(subset=["tri"]).set_index("date")["tri"].sort_index().resample(MONTH_END).last().pct_change()

uni = panel[panel["in_universe"] & panel["ret"].notna() & (panel["me_lag"] > 0)].copy()
vw = (uni["ret"] * uni["me_lag"]).groupby(uni["date"]).sum() / uni.groupby("date")["me_lag"].sum()

size_rows = []
for F, grp in chars.groupby("formation_date"):
    med = grp["me_formation"].median()
    small = set(grp.loc[grp["me_formation"] <= med, "symbol"])
    u = uni[uni["formation_date"] == F]
    w = lambda x: (x["ret"] * x["me_lag"]).sum() / x["me_lag"].sum()
    s = u[u["symbol"].isin(small)].groupby("date").apply(w)
    b = u[~u["symbol"].isin(small)].groupby("date").apply(w)
    size_rows.append((s - b).rename("size_spread"))
size_spread = pd.concat(size_rows).sort_index()

if (TRI_DIR / "nifty500.csv").exists():
    idx = pd.DataFrame({n: tri_returns(n) for n in ["nifty500", "nifty100", "midcap150", "smallcap250"]})
    cmp = pd.DataFrame({"ours_vw": vw, "nifty500": idx["nifty500"], "ours_size": size_spread,
                        "index_size": (idx["midcap150"] + idx["smallcap250"]) / 2 - idx["nifty100"]}).dropna()
    n_years = len(cmp) / 12
    cagr = (1 + cmp[["ours_vw", "nifty500"]]).prod() ** (1 / n_years) - 1
    te = (cmp["ours_vw"] - cmp["nifty500"]).std() * np.sqrt(12)
    print(f"\n6) Market, {cmp.index.min():%b-%Y} to {cmp.index.max():%b-%Y}: correlation "
          f"{cmp['ours_vw'].corr(cmp['nifty500']):.3f} | CAGR ours {cagr['ours_vw']:.2%} vs Nifty 500 TRI "
          f"{cagr['nifty500']:.2%} | tracking error {te:.2%} a year")
    print(f"7) Size spread: ours {cmp['ours_size'].mean():.2%} a month vs index {cmp['index_size'].mean():.2%} "
          f"| correlation {cmp['ours_size'].corr(cmp['index_size']):.3f}")
    CHECK_MARKET = {"corr": round(float(cmp["ours_vw"].corr(cmp["nifty500"])), 4),
                    "cagr_ours": round(float(cagr["ours_vw"]), 4), "cagr_index": round(float(cagr["nifty500"]), 4),
                    "size_ours": round(float(cmp["ours_size"].mean()), 5),
                    "size_index": round(float(cmp["index_size"].mean()), 5)}
else:
    print("\n6-7) Skipped: NSE TRI files not found in", TRI_DIR)
    CHECK_MARKET = {}

1) Universe coverage per June:
                members  with_me  with_bm  with_op  with_inv  financials
formation_date                                                          
2016-06-30          500      498      469      424       444          61
2017-06-30          500      496      468      421       462          66
2018-06-30          500      499      472      422       466          78
2019-06-30          500      500      471      417       469          81
2020-06-30          500      499      475      422       477          79
2021-06-30          500      499      474      415       473          84
2022-06-30          500      499      479      427       485          82
2023-06-30          500      500      486      430       481          88
2024-06-30          500      500      490      432       489          90
2025-06-30          500      499      491      434       487          92
2026-06-30          500      499      496      437       490         101

2) Share-count sour

,symbol,date,shares,shares_change,shares_source
1143,ABFRL,2016-03-31,7.690000e+08,7.268817,screener
5194,ALLCARGO,2025-03-31,1.500000e+09,0.525061,screener
5206,ALLCARGO,2026-03-31,1.500000e+09,0.526279,screener
5224,ALOKINDS,2020-12-31,4.965240e+09,2.605006,yfinance
5926,ANDHRABANK,2018-03-31,1.199000e+09,0.758624,screener
5938,ANDHRABANK,2019-03-31,2.884000e+09,1.411432,screener
10964,BAJAJHIND,2026-03-31,2.304074e+09,0.851435,yfinance
12513,BANKBARODA,2019-12-31,4.073140e+09,0.538997,yfinance
12694,BANKINDIA,2019-03-31,2.696640e+09,0.545804,yfinance
13625,BCG,2021-12-31,1.042520e+09,0.519241,yfinance



4) Largest stocks by June ME:
2016-06-30 TCS (502,961), RELIANCE (314,241), HDFCBANK (298,041), ITC (296,684), INFY (267,879), HDFC (199,263), COALINDIA (197,691), HINDUNILVR (194,087), ONGC (185,066), SUNPHARMA (183,659)
2019-06-30 TCS (835,730), RELIANCE (742,613), HDFCBANK (666,848), HINDUNILVR (387,087), HDFC (377,582), ITC (335,666), SBIN (322,402), INFY (313,807), KOTAKBANK (281,964), ICICIBANK (280,312)
2022-06-30 RELIANCE (1,768,946), TCS (1,205,723), HDFCBANK (748,524), INFY (614,461), HINDUNILVR (524,608), ICICIBANK (491,788), SBIN (424,214), BHARTIARTL (403,742), HDFC (393,894), ITC (340,817)
2025-06-30 RELIANCE (2,031,467), HDFCBANK (1,534,730), TCS (1,252,583), BHARTIARTL (1,204,331), ICICIBANK (1,031,551), SBIN (732,223), INFY (663,885), LICI (615,427), BAJFINANCE (581,468), HINDUNILVR (539,166)

5) Universe stocks that stopped trading: 75 | handled in MERGERS: 29
symbol
ADVANTA       2016-07-29
SBT           2017-03-15
SBBJ          2017-03-15
MYSOREBANK    2017-03-15
S

## Findings: Phase 1 v2 (run of 2-Nov-2026, data to 31-Oct-2026)

**Universe:** 500 members every June from 2016 to 2026 (NSE ranking files; 2025–2026 from six-month averages), about 880 distinct stocks over the decade, including delisted and merged ones. Coverage per June: June ME for 496–500 members, B/M 468–496, OP 415–437, investment 444–490; financial firms 61–101.

**Survivorship bias removed:** both checks against survivorship-free NSE indices now agree.

| Check (Jul-2016 to Jun-2026) | Phase 1 v1 (today's list) | Phase 1 v2 (point-in-time) |
|---|---|---|
| Market vs Nifty 500 TRI | CAGR 9.8% vs 13.9%, tracking error 4.6% | **CAGR 14.06% vs 13.90%, correlation 0.992, tracking error 2.09%** |
| Size spread vs (Midcap 150 + Smallcap 250)/2 − Nifty 100 | 1.30% vs 0.40% a month | **0.29% vs 0.40% a month, correlation 0.936** |

**Size sanity:** the largest stocks by June ME are the familiar large caps in every year (June 2016: TCS, RELIANCE, HDFCBANK, ITC, INFY; June 2025: RELIANCE, HDFCBANK, TCS, BHARTIARTL, ICICIBANK).

**Corporate actions:** all 29 mergers into listed stocks are credited with the share swap (e.g. HDFC → HDFC Bank, the 2017/2019/2020 PSU bank mergers, MINDTREE → LTI, TATASTLLP and TATASTLBSL → Tata Steel, TATACOFFEE → Tata Consumer, IDFC → IDFC First Bank); every ratio is confirmed by market prices (swap return within ±25%). Four renames missing from NSE's list were added (PHILIPCARB → PCBL, BURGERKING → RBA, WABCOINDIA → ZFCVINDIA, RUCHI → PATANJALI). Remaining delistings (insolvencies such as DHFL, JETAIRWAYS, RELCAPITAL; buy-outs such as HEXAWARE; mergers with cash/preference consideration such as CAIRN; TV18BRDCST → NETWORK18, ratio unconfirmed) have no delisting return.

**Share counts:** yfinance for most stock-months, checked month by month against NSE/Screener anchors (675 yfinance months in 126 stocks were more than 25% off and rejected; Screener counts dropped for 10 stocks whose median gap from NSE exceeds 10%). The remaining share jumps are mostly real issuance (e.g. the 2019–20 PSU bank recapitalisations, ALOKINDS's insolvency resolution).

In [38]:
# ==========================================================
# STEP 12: Save outputs and the run manifest
# ==========================================================
chars.to_csv(PROCESSED_DIR / "characteristics_annual.csv", index=False)
panel.to_csv(PROCESSED_DIR / "monthly_panel.csv", index=False)

manifest = {
    "run_date": RUN_DATE.isoformat(), "end_date": END_DATE.isoformat(),
    "last_price_date": str(LAST_PRICE_DATE.date()), "screener_snapshot": SCREENER_SNAPSHOT,
    "universe_size": UNIVERSE_SIZE, "formation_years": [int(y) for y in file_for_year],
    "ranking_file_per_year": {int(y): str(d.date()) for y, d in file_for_year.items()},
    "stocks_ever_in_universe": len(UNIVERSE_SIDS), "stocks_with_fundamentals": int(final_df["symbol"].nunique()),
    "rf_source": RF_SOURCE, "checks_step5": CHECKS_STEP5, "checks_market_size": CHECK_MARKET,
    "panel_rows": len(panel), "characteristics_rows": len(chars),
}
(PROCESSED_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Saved: universe.csv, monthly_prices.csv, final_fin_data.csv, rf_monthly.csv,")
print("       characteristics_annual.csv, monthly_panel.csv, run_manifest.json")
print(json.dumps(manifest, indent=2))

Saved: universe.csv, monthly_prices.csv, final_fin_data.csv, rf_monthly.csv,
       characteristics_annual.csv, monthly_panel.csv, run_manifest.json
{
  "run_date": "2026-11-12",
  "end_date": "2026-10-31",
  "last_price_date": "2026-10-30",
  "screener_snapshot": "2026-10-14",
  "universe_size": 500,
  "formation_years": [
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
    2026
  ],
  "ranking_file_per_year": {
    "2016": "2016-03-31",
    "2017": "2017-03-31",
    "2018": "2018-03-31",
    "2019": "2019-03-31",
    "2020": "2020-03-31",
    "2021": "2021-03-31",
    "2022": "2022-03-31",
    "2023": "2023-03-31",
    "2024": "2024-03-28",
    "2025": "2024-12-31",
    "2026": "2025-12-31"
  },
  "stocks_ever_in_universe": 879,
  "stocks_with_fundamentals": 844,
  "rf_source": "FRED INDIR3TIB01STM (3-month interbank rate, proxy for the 91-day T-bill)",
  "checks_step5": {
    "split/bonus events confirmed by the price drop": 469,
 

## Revision log: Phase 1 v1 → v2 (what was wrong, how it was found, how it is fixed)

Phase 4 validation (Steps 2–2.6: IIMA factors, Nifty total-return indices) showed that Phase 1 v1 had data errors large enough to change the results. Phase 1 was rebuilt as v2. **All Phase 2–4 results produced from v1 are superseded** and must be regenerated from v2 outputs.

To avoid confusion, old version is replaced with this v2 version.

### 1. Survivorship bias (universe)

| | |
|---|---|
| **Problem** | The universe was the Nifty 500 list downloaded on 14-Oct-2026, followed back to 2016. All 468 sample stocks were today's constituents; stocks that were dropped, delisted or merged (e.g. HDFC Ltd) were missing. |
| **Evidence** | Our SMB earned 15.6% a year; survivorship-free index spreads (Smallcap 250 / Midcap 150 − Nifty 100) earned 4.0–5.5% a year (gap 0.84–0.97% a month, NW t = 3.9–4.5). IIMA's SMB averaged −0.20% a month on the same months against our +1.25%. Our SMB also did not fall in the 2018–20 small-cap crash, unlike every benchmark. |
| **Fix** | Point-in-time universe: each June, the **top 500 NSE stocks by full market cap** from NSE's official ranking files (the latest file dated on or before 31 March, so no look-ahead), limited to stocks that traded in the previous 3 months. Members keep membership for the whole holding year even if they delist. |

### 2. Prices and returns

| Problem | Evidence | Fix |
|---|---|---|
| Returns assumed NSE's `prevclose` is adjusted on split/bonus ex-dates. It is not. | `prevclose` unadjusted in 303 of 306 events checked; fake monthly crashes, e.g. RELIANCE Sep-2017 (−51%), BAJFINANCE Jun-2025 (−90%), NYKAA Nov-2022 (−85%), JUBLFOOD Apr-2022 (−79%), VBL, GICRE, AIIL. | Daily return = (close × split factor + dividend) × rights factor ÷ **yesterday's actual close** − 1, with split/bonus factors from NSE's corporate-actions file. |
| Prices came from per-symbol downloads for today's 500 stocks only. | Delisted and dropped stocks had no prices. | Prices for every stock from **NSE daily bhavcopies** (both file formats), cached and read incrementally. |
| Renamed stocks were split into separate series or mismatched with their events. | E.g. AEGISCHEM → AEGISLOG, MINDAIND → UNOMINDA (an event matched four weeks late), MOTHERSUMI → MOTHERSON. | NSE symbol-change list; every record is mapped by **its own date** to the stock's latest symbol (permanent ID). Four renames missing from NSE's list were added (`MANUAL_RENAMES`: PHILIPCARB → PCBL, BURGERKING → RBA, WABCOINDIA → ZFCVINDIA, RUCHI → PATANJALI). |
| Mergers into a listed company were missing. | HDFC Ltd (a top-5 stock) absent from the whole sample. | Merged stocks stay in for their member years; the final month includes the **share-swap value** (`MERGERS` table, e.g. HDFC → HDFC Bank, 42:25). |

### 3. Corporate actions

| Problem | Fix |
|---|---|
| NSE's corporate-actions file is incomplete or wrong for some events. | 14 splits/bonuses added manually (e.g. ASIANPAINT 2013 ×10, SBIN 2014 ×10, JSWSTEEL 2017 ×10) and 2 factors fixed (HINDZINC 2011, MINDACORP 2015), all with factors implied by the one-day price drop. |
| Demergers looked like crashes. | Demerger ex-day return set to 0 (value moved to the new company): 22 found from price drops plus every demerger labelled in the NSE file (e.g. RELIANCE–Jio Financial 2023, ITC–ITC Hotels 2025, TATAMOTORS 2025). |
| Rights issues were ignored. | CRSP adjustment: return factor = cum price ÷ theoretical ex-rights price; factors above 1.5 used only when the price drop confirms them (IDEA 2019, M&MFIN 2020, NCC 2014). |
| Dividend parser read non-dividend amounts in combined subjects. | Only the dividend part is read (e.g. "Dividend Rs 10 / Split from Rs 10 to Re 1" was 21, now 10; "Dividend Rs 2 / Rights @ Rs 40" was 42, now 2). |
| "Purpose Revised" duplicates double-counted dividends. | Each (stock, ex-date, amount) counted once. |
| Dividend and split on the same ex-date. | Dividend treated as per **old** share. |
| No systematic check of events. | Automatic handling, logged in Step 5.9: a split/bonus whose price drop falls within 15 days of the NSE ex-date is moved to that day, and one with no matching drop is dropped; a one-day move > 37.5% down or > 66% up that matches a standard split/bonus factor (within 3%) is added as a missing event, and any other is kept as a real move; rights factors > 1.5 are used only when the price drop confirms them. **Must be 0:** manual demergers not mapped to a trading day. One-day falls of 20–37.5% were also reviewed: apart from two missed bonuses they were real news-driven moves and are kept. |

### 4. Share counts and market cap (ME)

| Problem | Evidence | Fix |
|---|---|---|
| Shares before 2017 were backfilled from yfinance using `ca_factor`, which was almost always 1 (because of the `prevclose` problem). Splits were never undone, so ME before a split used the post-split share count. | June 2016 ME overstated: BAJFINANCE ~9×, JSWSTEEL ~9×, BEL ~10×, GRASIM ~5×, ONGC ~1.5×; Bajaj Finance appeared as India's largest company. | Shares carried in **split-free units** through split/bonus events from the corporate-actions tables. |
| yfinance gaps left large stocks without ME. | TCS had no June ME in 2016 and 2018 (dropped from both formations) and ~2× ME in June 2017. | Three sources in priority order: **yfinance → NSE market cap ÷ close → Screener equity capital ÷ face value**, gaps filled in split-free units. |
| No cross-check between share sources. | — | Screener counts are dropped for a stock if their median gap from NSE exceeds 10%; a yfinance month is rejected if it is more than 25% from the NSE/Screener anchor; yfinance is ignored in months with a split/bonus (its monthly median mixes pre- and post-event counts). |

### 5. Fundamentals and flags

| Problem | Fix |
|---|---|
| Fundamentals only for today's 500 stocks. | Screener pages for every stock ever in the universe, including delisted and merged ones (Screener keeps them, e.g. HDFC Ltd). Stocks with no fundamentals stay in MKT and the size sort only (FF rule). |
| Financial flag and sector taken from today's NSE list. | Screener's "Broad Sector" (NSE classification), with today's NSE list as fallback; `is_financial` = "Financial Services". |
| Markdown said financials were excluded from all four characteristic sorts. | Corrected to the Phase 2 rule: excluded from the **RMW sort only**. |

### 6. Reproducibility and pipeline

| Problem | Fix |
|---|---|
| Step 0 deleted `data/interim` and `data/processed` on every run (this wiped Phase 2–4 outputs on kernel restarts). | Deletion only when `CLEAN_DERIVED` is switched on. |
| The data cutoff moved with today's date and triggered silent downloads; the Screener snapshot changed with the run date. | All settings in one config cell, overridable by environment variables for CI/CD (`FF_END_DATE`, `FF_SCREENER_SNAPSHOT`, `FF_CLEAN_DERIVED`; `FF_UNIVERSE_SIZE` for testing only); Screener snapshot reused while younger than 90 days; every run writes `run_manifest.json`. |
| The Rf proxy was re-downloaded each run and documented as a temporary 91-day T-bill. | FRED 3-month interbank rate (validated vs IIMA's Rf: correlation 0.96, gap ~0.16% a year) saved once and reused; documented as the Rf source unless the RBI T-bill file is supplied. |
| No Phase 1 check against independent benchmarks. | New final checks: value-weighted market return vs the **Nifty 500 TRI** (v1: CAGR 9.8% vs 13.9%, tracking error 4.6% a year), and small-minus-big return vs the **Midcap/Smallcap − Nifty 100 spread**. |

See **Limitations** in the Phase 1 summary below.

## Phase 1 summary

**Outputs** (`data/processed/`): `universe.csv`, `monthly_panel.csv`, `characteristics_annual.csv`, `monthly_prices.csv`, `final_fin_data.csv`, `rf_monthly.csv`, `run_manifest.json`; corporate-action tables in `data/interim/`.

**Notes for Phase 2**
- Use only `in_universe == True` rows (panel) and the rows of `characteristics_annual.csv` (all are members). `symbol` is each stock's latest NSE symbol.
- Returns already include dividends and every corporate-action adjustment: **do not apply another split correction in Phase 2.**
- `shares_source` values: yfinance, nse, screener, carried forward, backfilled, missing.
- Financial firms (`is_financial`): excluded from the RMW sort only; included in MKT, SMB, HML, CMA. BE > 0 required for the HML and RMW sorts.
- Eligibility thresholds (`months_traded`, `tv_median_12m`, `trade_days_12m`) are set in Phase 2.

**Limitations**
- Universe = top 500 by NSE's full market cap ranking; the official Nifty 500 also applies liquidity/F&O eligibility rules and reviews membership in March and September. Ranking files are annual (2016–Mar 2019: rank only; Dec 2019–Mar 2024: market cap on the date; from Dec 2024: six-month averages).
- Fundamentals start FY2015 (Screener), so the first full five-factor year is July 2016.
- Delistings other than listed mergers have no delisting return (returns stop at the last trade).
- Share counts between data points ignore issues/buybacks; Screener rounds equity capital to ₹ crore.
- Rights issues treated as fully paid; small demergers before ~2019 may be missed; neutralising a demerger day after a trading gap also removes that gap's real price move.
- Rf: FRED 3-month interbank rate unless the RBI T-bill file is supplied.

**Data refresh (no code changes):** run all cells. Once a year, add the new NSE ranking file to `MCAP_RANKINGS` (Step 0). Rows in the manual tables are needed only if Step 5.9 shows manual demergers not mapped > 0 (fix in 5.3), Step 6 warns about a merger swap beyond ±25% (fix the ratio in `MERGERS`), or Step 11 check 5b lists a real rename (add it to `MANUAL_RENAMES` in Step 1).